# Docling PDF 파싱 결과 평가 (Parsing Evaluation)

이 노트북은 **PDF를 파싱하는 노트북이 아니다.** `docling_parsing_test.ipynb`가 이미 만들어 둔
파싱 산출물(`output/`)을 **원본 PDF 및 Ground Truth와 대조하여 구조 복원 품질을 채점**한다.

| # | 평가 영역 | 핵심 질문 | 가중치 |
|---|---|---|---|
| 1 | **Layout Analysis** | PDF 페이지의 영역을 올바른 문서 요소로 구분하는가? | 0.30 |
| 2 | **Text Hierarchy** | 문서의 논리적 계층(H1→H2→H3→본문)과 순서를 복원하는가? | 0.30 |
| 3 | **Table Structure** | 표의 행·열·헤더·셀·병합 구조와 값을 복원하는가? | 0.40 |

## Ground Truth

`tiger_inc/`에는 **같은 파일명의 `md/`와 `pdf/`가 쌍으로 존재**한다. PDF는 이 마크다운 원고를
조판해 만든 것이므로, **`tiger_inc/md/*.md`가 해당 PDF의 정답지(GT)** 다. 평가는 전부 이 GT와
docling 산출물을 대조하는 방식이며, **docling 결과를 docling으로 채점하지 않는다.**

| 구분 | 문서 | GT | 처리 |
|---|---|---|---|
| 사내 규정 `tiger_inc/pdf/` | 8종 | ✅ `tiger_inc/md/` 동일 파일명 | **자동 정량 평가** |
| 법령 `tiger_inc/law/` | 3종 | ❌ 없음 | **N/A — 점수 산출에서 제외**, 사람 검수용 시트만 생성 |

## 채점 원칙 (반드시 지킴)

1. **GT가 없으면 점수를 만들지 않는다.** 해당 지표는 `N/A`로 남기고, 가중치는 나머지 지표로
   재정규화한다(무엇이 빠졌는지 리포트에 명시).
2. **정답 Bounding Box가 없다.** 마크다운 GT에는 좌표가 없으므로 **IoU는 산출하지 않고 `N/A`**로
   두며, 대신 기하 이상치 탐지 + 사람 검수 시트(`layout_manual_review.csv`)로 대체한다(§3-3).
3. **병합 셀(rowspan/colspan)은 마크다운으로 표현할 수 없다.** 따라서 병합 정확도도 `N/A`이며,
   GT의 값 반복 패턴을 이용한 **휴리스틱 대조는 참고 정보로만** 출력하고 점수에 넣지 않는다.

## 입력 / 출력

```text
입력  tiger_inc/{pdf,law}/*.pdf          원본 PDF (페이지·문자 수 확인)
      tiger_inc/md/*.md                  Ground Truth
      docling_eval/output/layout/layout_result.csv     docling 요소 전수(리딩오더·bbox·타입)
      docling_eval/output/tables/<문서>/table_NN.json  docling 표 셀 격자
      docling_eval/output/markdown/<문서>.md           docling 마크다운

출력  docling_eval/output/evaluation/evaluation_summary.csv
                                        /evaluation_details.csv
                                        /error_cases.csv
                                        /qualitative_review.csv
                                        /layout_manual_review.csv
                                        /evaluation_report.md
```

> **선행 조건** — `docling_parsing_test.ipynb`를 먼저 실행해 `output/`이 채워져 있어야 한다.
> 커널은 동일하게 conda env `docling`을 쓴다(§CLAUDE.local.md). 이 노트북 자체는 docling을
> import 하지 않으므로 pandas만 있으면 돌아간다.

---
## 1. 평가 환경 및 데이터 로드

In [1]:
from __future__ import annotations

import json
import re
import sys
import unicodedata
from collections import Counter
from difflib import SequenceMatcher
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 200)


def _resolve_base_dir() -> Path:
    """CWD가 레포 루트든 docling_eval 내부든 같은 docling_eval 을 가리키게 한다."""
    cwd = Path.cwd().resolve()
    for cand in (cwd, *cwd.parents):
        if cand.name == "docling_eval":
            return cand
        if (cand / "docling_eval").is_dir() or (cand / "tiger_inc").is_dir():
            return cand / "docling_eval"
    return cwd / "docling_eval"


BASE_DIR = _resolve_base_dir()
REPO_ROOT = BASE_DIR.parent

# ── 입력 ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = BASE_DIR / "output"
LAYOUT_CSV = OUTPUT_DIR / "layout" / "layout_result.csv"     # docling 요소 전수
DOCLING_TABLE_DIR = OUTPUT_DIR / "tables"                    # docling 표 격자
DOCLING_MD_DIR = OUTPUT_DIR / "markdown"                     # docling 문서별 마크다운
DOCLING_MD_PATH = OUTPUT_DIR / "docling_result.md"           # 합본(참고용)
GT_MD_DIR = REPO_ROOT / "tiger_inc" / "md"                   # ← Ground Truth
PDF_DIRS = [REPO_ROOT / "tiger_inc" / "pdf", REPO_ROOT / "tiger_inc" / "law"]

# ── 출력 ────────────────────────────────────────────────────────────────────
EVAL_DIR = OUTPUT_DIR / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# ▼▼▼ 평가 대상 — 바꿀 곳은 여기뿐 ▼▼▼
TARGET_DOCS = None          # None = GT가 있는 전체. 일부만: ["법인카드_사용규정"]
# ▲▲▲

# ── 채점 파라미터 ───────────────────────────────────────────────────────────
MATCH_THRESHOLD = 0.75      # 요소 텍스트 유사도 매칭 임계값
TABLE_MATCH_THRESHOLD = 0.30  # 표 매칭(셀 텍스트 자카드) 임계값
AREA_WEIGHTS = {"Layout": 0.30, "Hierarchy": 0.30, "Table": 0.40}

if not LAYOUT_CSV.exists():
    raise FileNotFoundError(
        f"docling 산출물이 없습니다: {LAYOUT_CSV}\n"
        "→ 먼저 docling_eval/docling_parsing_test.ipynb 를 끝까지 실행하세요."
    )
if not GT_MD_DIR.is_dir():
    raise FileNotFoundError(f"Ground Truth 폴더가 없습니다: {GT_MD_DIR}")

print(f"python     : {sys.version.split()[0]}")
print(f"pandas     : {pd.__version__}")
print(f"BASE_DIR   : {BASE_DIR}")
print(f"GT (정답)  : {GT_MD_DIR}")
print(f"평가 산출물: {EVAL_DIR}")

python     : 3.12.13
pandas     : 3.0.5
BASE_DIR   : D:\project\SKN29-FINAL-1TEAM\docling_eval
GT (정답)  : D:\project\SKN29-FINAL-1TEAM\tiger_inc\md
평가 산출물: D:\project\SKN29-FINAL-1TEAM\docling_eval\output\evaluation


### 1-1. 텍스트 정규화 규칙

한글 PDF 파싱 비교에서는 정규화 수준을 **두 단계로 나누는 것이 핵심**이다.

| 함수 | 처리 | 용도 |
|---|---|---|
| `norm_strict` | NFKC · 마크다운 강조 제거 · 연속 공백 1칸 | **값 일치** 판정(셀 내용 등) |
| `norm_loose` | `norm_strict` + **모든 공백 제거** + 구두점 제거 | **요소 매칭**(대응 관계 찾기) |

둘을 나누는 이유: docling은 양끝맞춤 조판 PDF에서 `타 이 거 주 식 회 사`, `최적 화` 처럼
**자간·어절 내부에 공백을 삽입**한다. 공백을 지우지 않으면 같은 문장이 서로 다른 요소로 잡혀
매칭이 통째로 무너진다. 반대로 공백을 지운 채로만 채점하면 이 결함이 **보이지 않게 되므로**,
`norm_loose`는 같은데 `norm_strict`가 다른 경우를 따로 세어 **`공백·자간 오류`** 로 분류한다(§7).

In [2]:
_MD_EMPH = re.compile(r"\*\*|__|\*|`|~~")
_WS = re.compile(r"\s+")
_PUNCT = re.compile(r"[\s.,·:;()\[\]{}\"'“”‘’!?/\\|~\-—–―_+=<>※•]")


def norm_strict(text: object) -> str:
    """값 일치 판정용 — 공백은 1칸으로 접되 지우지는 않는다."""
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return ""
    s = unicodedata.normalize("NFKC", str(text))
    s = _MD_EMPH.sub("", s)
    return _WS.sub(" ", s).strip()


def norm_loose(text: object) -> str:
    """요소 매칭용 — 공백·구두점을 전부 지운다(CJK 자간 결함 흡수)."""
    return _PUNCT.sub("", norm_strict(text).lower())


def similarity(a: str, b: str) -> float:
    """norm_loose 문자열 간 유사도(0~1)."""
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    if a == b:
        return 1.0
    return SequenceMatcher(None, a, b, autojunk=False).ratio()


def spacing_only_diff(gt: str, dt: str) -> bool:
    """내용은 같은데 공백/자간만 다른가 (= 조판 유래 텍스트 결함)."""
    return norm_strict(gt) != norm_strict(dt) and norm_loose(gt) == norm_loose(dt)


assert norm_loose("타 이 거 주 식 회 사") == norm_loose("타이거주식회사")
assert spacing_only_diff("비용 최적 화", "비용 최적화")
assert not spacing_only_diff("1,500,000", "1,500,00")
print("정규화 유틸 OK")

정규화 유틸 OK


### 1-2. Ground Truth 마크다운 파서

GT `.md`를 **요소 시퀀스**로 분해한다. 이것이 모든 채점의 기준선이다.

| 마크다운 | GT 요소 타입 | 비고 |
|---|---|---|
| `# ~ ######` | `Heading` (`Level` = `#` 개수) | 문서 제목 `#` 포함 |
| `\| a \| b \|` 연속 블록 | `Table` | 구분선(`\|---\|`) 위 행을 헤더로 |
| `- ` `* ` `1. ` | `List` | |
| `> ` · 일반 문단 | `Text` | 인용도 본문으로 본다 |
| `---` · 코드펜스 · 빈 줄 | (제외) | 구분선은 요소가 아님 |

여러 줄에 걸친 문단은 **빈 줄 기준으로 하나의 요소**로 합친다(PDF의 문단 단위와 맞추기 위함).

In [3]:
_H_RE = re.compile(r"^(#{1,6})\s+(.*)$")
_LI_RE = re.compile(r"^\s{0,3}(?:[-*+]\s+|\d+[.)]\s+)(.*)$")
_TABLE_SEP_RE = re.compile(r"^\|[\s:|-]+\|$")


def _split_pipe_row(line: str) -> list[str]:
    return [c.strip() for c in line.strip().strip("|").split("|")]


def parse_gt_markdown(path: Path, doc_name: str) -> tuple[list[dict], list[dict]]:
    """GT .md → (요소 시퀀스, 표 목록).

    표는 요소 시퀀스에 `Table` 한 줄로 들어가고, 격자 자체는 별도 목록으로 돌려준다.
    """
    lines = path.read_text(encoding="utf-8").splitlines()
    elements: list[dict] = []
    tables: list[dict] = []
    buf: list[str] = []
    tbl_buf: list[str] = []
    in_code = False

    def flush_text() -> None:
        if not buf:
            return
        text = " ".join(x.strip() for x in buf).strip()
        buf.clear()
        if text:
            elements.append({"Type": "Text", "Level": None, "Text": text})

    def flush_table() -> None:
        if not tbl_buf:
            return
        rows = [_split_pipe_row(r) for r in tbl_buf if not _TABLE_SEP_RE.match(r)]
        has_sep = any(_TABLE_SEP_RE.match(r) for r in tbl_buf)
        tbl_buf.clear()
        if not rows:
            return
        ncols = max(len(r) for r in rows)
        grid = [r + [""] * (ncols - len(r)) for r in rows]
        idx = len(tables) + 1
        tables.append({
            "Document": doc_name, "Table": idx,
            "Rows": len(grid), "Cols": ncols,
            "HeaderRows": 1 if has_sep else 0,
            "grid": grid,
        })
        flat = " ".join(c for row in grid for c in row)
        # NormSrc: 표는 셀 내용으로 매칭한다(치수 표기는 매칭 노이즈라 제외).
        elements.append({"Type": "Table", "Level": None, "TableIdx": idx,
                         "Text": f"<table {len(grid)}x{ncols}> {flat}", "NormSrc": flat})

    for raw in lines:
        line = raw.rstrip()
        stripped = line.strip()

        if stripped.startswith("```"):
            in_code = not in_code
            continue
        if in_code:
            continue

        if stripped.startswith("|") and stripped.endswith("|") and stripped.count("|") >= 2:
            flush_text()
            tbl_buf.append(stripped)
            continue
        flush_table()

        if not stripped:
            flush_text()
            continue
        if set(stripped) <= set("-*_") and len(stripped) >= 3:   # 수평선
            flush_text()
            continue

        m = _H_RE.match(stripped)
        if m:
            flush_text()
            elements.append({"Type": "Heading", "Level": len(m.group(1)),
                             "Text": m.group(2).strip()})
            continue

        m = _LI_RE.match(line)
        if m:
            flush_text()
            elements.append({"Type": "List", "Level": None, "Text": m.group(1).strip()})
            continue

        if stripped.startswith(">"):
            flush_text()
            elements.append({"Type": "Text", "Level": None,
                             "Text": stripped.lstrip("> ").strip()})
            continue

        buf.append(stripped)

    flush_text()
    flush_table()

    for i, el in enumerate(elements, start=1):
        el["Document"] = doc_name
        el["Order"] = i
        el["norm"] = norm_loose(el.pop("NormSrc", el["Text"]))
    return elements, tables


print("GT 파서 정의 완료")

GT 파서 정의 완료


### 1-3. docling 산출물 로더 및 대상 문서 확정

docling 요소는 `output/layout/layout_result.csv`(리딩오더 전수, bbox 포함)에서 읽는다.
GT 마크다운에는 **머리말/꼬리말·이미지가 없으므로**(조판 과정에서 생김) 요소 대조에서는
`Header`/`Footer`를 제외하고, 대신 그 개수를 따로 보고한다. `Picture`는 GT로 참/거짓을
확정할 수 없어 **`GT 불가` 버킷**으로 분리한다.

| docling 요소 타입 | 평가 타입 | 사유 |
|---|---|---|
| `Title`, `Heading` | `Heading` | |
| `Text`, `Caption`, `Footnote`, `Formula`, `Code`, `Reference` | `Text` | GT md에서는 모두 본문 문단 |
| `List` | `List` | |
| `Table` | `Table` | |
| `Header`, `Footer` | (제외) | GT에 대응물이 없음 — 개수만 보고 |
| `Picture` | (GT 불가) | GT md에 도해가 없어 정오 판정 불가 |

**표 요소의 텍스트 보강** — `layout_result.csv`의 `Table` 행은 본문 대신 `<table 3x2>`나 캡션만
들고 있어, GT 표(셀 내용이 펼쳐진 문자열)와 절대 매칭되지 않는다. 이대로 두면 표가 §3에서
전부 `Missing`+`Extra`로 이중 계상된다. 그래서 표 요소는 **`table_NN.json` 격자를 펼쳐 매칭용
텍스트로 붙인다**(리딩오더의 k번째 `Table` 요소 ↔ `table_(k+1).json` — 문서별로 개수·치수가
일치함을 확인했다). 격자 로더는 §5에서도 그대로 재사용한다.

In [4]:
EVAL_TYPE = {
    "Title": "Heading", "Heading": "Heading",
    "Text": "Text", "Caption": "Text", "Footnote": "Text",
    "Formula": "Text", "Code": "Text", "Reference": "Text", "Form": "Text",
    "List": "List",
    "Table": "Table",
}
FURNITURE_TYPES = {"Header", "Footer"}      # GT에 대응물 없음 → 대조 제외
UNVERIFIABLE_TYPES = {"Picture"}            # GT로 정오 판정 불가

layout_df = pd.read_csv(LAYOUT_CSV, encoding="utf-8-sig")
layout_df.columns = [c.strip() for c in layout_df.columns]
layout_df["Text"] = layout_df["Text"].fillna("")

docling_docs = sorted(layout_df["Document"].unique())
gt_paths = {p.stem: p for p in sorted(GT_MD_DIR.glob("*.md"))}
pdf_paths = {p.stem: p for d in PDF_DIRS if d.is_dir() for p in sorted(d.glob("*.pdf"))}

scored_docs, na_docs = [], []
for name in docling_docs:
    if name in gt_paths and (TARGET_DOCS is None or name in TARGET_DOCS):
        scored_docs.append(name)
    elif name not in gt_paths:
        na_docs.append(name)

GT_ELEMENTS: dict[str, list[dict]] = {}
GT_TABLES: dict[str, list[dict]] = {}
for name in scored_docs:
    els, tbls = parse_gt_markdown(gt_paths[name], name)
    GT_ELEMENTS[name] = els
    GT_TABLES[name] = tbls


def load_docling_tables(name: str) -> list[dict]:
    """docling 표 JSON → 조밀 격자. 병합 셀은 앵커 하나만 저장되므로 span만큼 채운다."""
    tdir = DOCLING_TABLE_DIR / name
    out = []
    if not tdir.is_dir():
        return out
    for path in sorted(tdir.glob("table_*.json")):
        raw = json.loads(path.read_text(encoding="utf-8"))
        rows, cols = int(raw["Rows"]), int(raw["Cols"])
        grid = [["" for _ in range(cols)] for _ in range(rows)]
        header_rows: set[int] = set()
        for c in raw.get("cells", []):
            for r in range(c["row"], min(rows, c["row"] + c.get("row_span", 1))):
                for k in range(c["col"], min(cols, c["col"] + c.get("col_span", 1))):
                    grid[r][k] = c["text"]
            if c.get("column_header"):
                header_rows.add(c["row"])
        out.append({
            "Document": name, "Table": int(raw["Table"]), "Page": raw.get("Page"),
            "Rows": rows, "Cols": cols, "grid": grid,
            "HeaderRows": len(header_rows),
            "HeaderDetected": raw.get("Header") == "Detected",
            "Merged": int(raw.get("Merged Cells", 0)),
            "Caption": raw.get("Caption", ""),
        })
    return out


DT_TABLES = {name: load_docling_tables(name) for name in docling_docs}


def docling_elements(name: str, drop_furniture: bool = True) -> list[dict]:
    """docling 요소 시퀀스(리딩오더). 평가 타입으로 매핑해 돌려준다."""
    sub = layout_df[layout_df["Document"] == name].sort_values("Order")
    grids = DT_TABLES.get(name, [])
    out, t_seen = [], 0
    for _, r in sub.iterrows():
        raw_type = str(r["Element Type"])
        if drop_furniture and raw_type in FURNITURE_TYPES:
            continue
        if raw_type in UNVERIFIABLE_TYPES:
            continue
        eval_type = EVAL_TYPE.get(raw_type, raw_type)
        norm_src = str(r["Text"])
        if raw_type == "Table":
            if t_seen < len(grids):     # k번째 Table 요소 ↔ table_(k+1).json
                norm_src = " ".join(c for row in grids[t_seen]["grid"] for c in row)
            t_seen += 1
        out.append({
            "Document": name,
            "Order": int(r["Order"]),
            "Page": int(r["Page"]) if pd.notna(r["Page"]) else None,
            "RawType": raw_type,
            "Type": eval_type,
            "Level": int(str(r["Level"])[1:]) if isinstance(r["Level"], str) and r["Level"].startswith("H") else None,
            "BBox": r.get("BBox (l,t,r,b)"),
            "Text": str(r["Text"]),
            "norm": norm_loose(norm_src),
        })
    return out


DT_ELEMENTS = {name: docling_elements(name) for name in scored_docs}

inventory = pd.DataFrame([
    {
        "Document": n,
        "PDF": "O" if n in pdf_paths else "-",
        "GT(md)": "O" if n in gt_paths else "-",
        "GT 요소": len(GT_ELEMENTS.get(n, [])) or None,
        "docling 요소": len(DT_ELEMENTS.get(n, [])) or None,
        "GT 표": len(GT_TABLES.get(n, [])) or None,
        "평가": "자동 정량" if n in scored_docs else "N/A (GT 없음 → 사람 검수)",
    }
    for n in docling_docs
])
print(f"자동 평가 대상 {len(scored_docs)}종 / GT 없음 {len(na_docs)}종")
display(inventory)

자동 평가 대상 8종 / GT 없음 3종


,Document,PDF,GT(md),GT 요소,docling 요소,GT 표,평가
0,법인세법,O,-,NaN,NaN,NaN,N/A (GT 없음 → 사람 검수)
1,법인카드_사용규정,O,O,98.0,102.0,2.0,자동 정량
2,부가가치세법,O,-,NaN,NaN,NaN,N/A (GT 없음 → 사람 검수)
3,부서소개,O,O,16.0,19.0,7.0,자동 정량
4,업무추진비_사용규정,O,O,80.0,86.0,3.0,자동 정량
5,여신전문금융업법,O,-,NaN,NaN,NaN,N/A (GT 없음 → 사람 검수)
6,조직도,O,O,22.0,26.0,2.0,자동 정량
7,조직설계_상세기획서,O,O,59.0,67.0,6.0,자동 정량
8,직급체계,O,O,19.0,21.0,2.0,자동 정량
9,출장비_사용규정,O,O,66.0,68.0,3.0,자동 정량


---
## 2. 원본 PDF 기본 정보 확인

채점 전에 **원본 PDF 자체**를 열어 페이지 수·크기·추출 가능 문자 수를 본다. 목적은 두 가지다.

- docling이 보고한 페이지 수와 실제 페이지 수가 같은지 (페이지 누락 여부)
- PDF에 **텍스트 레이어가 있는지** — 문자가 0에 가까우면 스캔본이라 OCR 없이는 평가 자체가 무의미하다

`pypdfium2`(docling 의존성)를 쓰되, 없으면 이 절만 건너뛴다.

> `Char Ratio`는 **참고 지표**다. PDF 원시 추출은 줄바꿈·공백을 그대로 세고 docling은 정리해서
> 내보내므로 1.00이 되지 않는다. 0.5 밑으로 떨어지면 텍스트를 크게 흘렸다는 신호로 본다.

In [5]:
try:
    import pypdfium2 as pdfium
    _HAS_PDFIUM = True
except Exception as exc:      # noqa: BLE001 — 없으면 이 절만 N/A
    _HAS_PDFIUM = False
    print(f"[warn] pypdfium2 없음 ({exc}) → PDF 원본 정보는 N/A")

pdf_rows = []
if _HAS_PDFIUM:
    for name in docling_docs:
        path = pdf_paths.get(name)
        if path is None:
            continue
        pdf = pdfium.PdfDocument(path)
        try:
            n_pages = len(pdf)
            w, h = pdf[0].get_size()
            chars = 0
            for i in range(n_pages):
                tp = pdf[i].get_textpage()
                chars += len(tp.get_text_range() or "")
        finally:
            pdf.close()
        sub = layout_df[layout_df["Document"] == name]
        dt_pages = int(sub["Page"].max())
        # 표 요소의 Chars 는 "<table 3x2>" 같은 자리표시자라 셀 텍스트로 대체해야 한다.
        dt_chars = int(sub.loc[sub["Element Type"] != "Table", "Chars"].sum())
        dt_chars += sum(len(c) for t in DT_TABLES.get(name, [])
                        for row in t["grid"] for c in row)
        pdf_rows.append({
            "Document": name,
            "PDF Pages": n_pages,
            "Docling Max Page": dt_pages,
            "Page Match": "OK" if n_pages == dt_pages else f"MISMATCH ({n_pages}≠{dt_pages})",
            "Size (pt)": f"{w:.0f}x{h:.0f}",
            "PDF Chars": chars,
            "Docling Chars": dt_chars,
            "Text Layer": "있음" if chars > 200 else "없음/희박 (스캔본 의심)",
        })

pdf_info_df = pd.DataFrame(pdf_rows)
if not pdf_info_df.empty:
    pdf_info_df["Char Ratio"] = (
        pdf_info_df["Docling Chars"] / pdf_info_df["PDF Chars"].replace(0, pd.NA)
    ).round(3)
    display(pdf_info_df)
    bad = pdf_info_df[pdf_info_df["Page Match"] != "OK"]
    print("페이지 수 불일치:", "없음" if bad.empty else list(bad["Document"]))
else:
    print("PDF 원본 정보: N/A")

,Document,PDF Pages,Docling Max Page,Page Match,Size (pt),PDF Chars,Docling Chars,Text Layer,Char Ratio
0,법인세법,97,97,OK,595x842,188243,180260,있음,0.958
1,법인카드_사용규정,7,7,OK,595x842,6148,5748,있음,0.935
2,부가가치세법,35,35,OK,595x842,65134,62009,있음,0.952
3,부서소개,5,5,OK,595x842,2633,2357,있음,0.895
4,업무추진비_사용규정,6,6,OK,595x842,6566,6227,있음,0.948
5,여신전문금융업법,34,34,OK,595x842,57222,53887,있음,0.942
6,조직도,6,6,OK,595x842,2663,1871,있음,0.703
7,조직설계_상세기획서,12,12,OK,595x842,8087,6867,있음,0.849
8,직급체계,4,4,OK,595x842,1648,1546,있음,0.938
9,출장비_사용규정,6,6,OK,595x842,4843,4555,있음,0.941


페이지 수 불일치: 없음


### 2-1. 텍스트 보존율 (참고 지표)

구조를 따지기 전에 **글자 자체가 얼마나 살아남았는지** 먼저 본다. GT 마크다운과 docling
마크다운(`output/markdown/<문서>.md`)의 **문자 다중집합**을 비교해 보존율/추가율을 낸다.

문자 단위 집합 비교라 순서·구조는 보지 않는다. 여기서 낮게 나오면 §3~§5의 구조 점수는
해석할 가치가 없으므로, **구조 평가의 전제 조건 확인용**으로만 쓴다(점수 미반영).

In [6]:
cov_rows = []
for name in scored_docs:
    dt_md = DOCLING_MD_DIR / f"{name}.md"
    if not dt_md.exists():
        continue
    gt_chars = Counter(norm_loose(gt_paths[name].read_text(encoding="utf-8")))
    dt_chars = Counter(norm_loose(dt_md.read_text(encoding="utf-8")))
    inter = sum((gt_chars & dt_chars).values())
    cov_rows.append({
        "Document": name,
        "GT Chars": sum(gt_chars.values()),
        "Docling Chars": sum(dt_chars.values()),
        "보존율(recall)": round(inter / sum(gt_chars.values()), 3) if gt_chars else None,
        "정밀도(precision)": round(inter / sum(dt_chars.values()), 3) if dt_chars else None,
    })

text_cov_df = pd.DataFrame(cov_rows)
if text_cov_df.empty:
    print(f"docling 마크다운을 찾지 못했습니다: {DOCLING_MD_DIR}")
else:
    display(text_cov_df)
    print(f"평균 문자 보존율 = {text_cov_df['보존율(recall)'].mean():.3f}  "
          "(구조 평가의 전제 확인용 — 점수에는 반영하지 않음)")

,Document,GT Chars,Docling Chars,보존율(recall),정밀도(precision)
0,법인카드_사용규정,4070,4064,0.994,0.996
1,부서소개,1593,1607,0.987,0.978
2,업무추진비_사용규정,4393,4385,0.991,0.993
3,조직도,1631,1191,0.717,0.982
4,조직설계_상세기획서,4973,4616,0.909,0.980
5,직급체계,976,951,0.961,0.986
6,출장비_사용규정,3145,3182,0.988,0.976
7,회식_운영규정,5512,5340,0.958,0.989


평균 문자 보존율 = 0.938  (구조 평가의 전제 확인용 — 점수에는 반영하지 않음)


---
## 3. Layout Analysis 평가

> **docling이 PDF의 영역을 올바른 문서 요소로 구분하는가?**

GT 요소 시퀀스와 docling 요소 시퀀스를 **텍스트로 정렬(alignment)** 한 뒤,

- 정렬된 쌍 → **탐지 성공(TP)**, 타입이 다르면 **분류 오류**
- 정렬되지 않은 GT 요소 → **Missing(FN)**
- 정렬되지 않은 docling 요소 → **Extra(FP)**

로 나눈다.

### 3-0. 정렬 엔진

두 단계로 맞춘다.

1. **순서 보존 블록 매칭** — `difflib.SequenceMatcher`를 요소 텍스트 리스트에 적용해 순서를
   유지하는 일치 구간을 먼저 확정한다.
2. **잔여 그리디 매칭** — 남은 요소끼리 유사도 `MATCH_THRESHOLD`(기본 0.75) 이상인 쌍을
   유사도 내림차순으로 1:1 배정한다. 이 단계에서 붙는 쌍은 **순서가 뒤바뀐 요소**(리딩오더 결함)나
   문단이 갈라진 요소다 — §4의 순서 정확도가 이걸 잡아낸다.
3. **포함 관계 구제** — 한쪽 텍스트가 다른 쪽에 통째로 들어 있고 길이 비가 0.5 이상이면 같은
   요소로 본다. `제2조 (정의)` ↔ `제2조 (정의) 개정 v1.1`처럼 **조판 배지가 헤딩에 섞여 들어온**
   경우가 여기 걸린다. 이건 "못 찾은 것"이 아니라 "찾았는데 텍스트가 오염된 것"이므로,
   미탐지로 몰아 이중으로 깎지 않고 **`Text Mismatch`** 오류로 따로 센다.

표 요소끼리는 임계값을 `TABLE_MATCH_THRESHOLD`(0.30)로 완화한다 — 셀 순서가 흔들리거나 표가
쪼개지면 문자열 유사도가 쉽게 0.75 밑으로 떨어지기 때문이다.

정렬은 Layout·Hierarchy·표 행 대조에 **모두 같은 함수를 쓴다.**

In [7]:
CONTAINMENT = 0.50          # 포함 관계 구제 — 짧은 쪽/긴 쪽 길이 비 하한


def align_sequences(gt_items: list[dict], dt_items: list[dict],
                    threshold: float = MATCH_THRESHOLD,
                    pair_threshold=None, containment: float = 0.0) -> dict:
    """GT/docling 요소 시퀀스를 1:1 정렬한다.

    pair_threshold(gt_item, dt_item) 를 주면 쌍마다 임계값을 다르게 쓸 수 있다
    (표끼리는 셀 순서가 흔들려 문자열 유사도가 낮게 나오므로 완화한다).

    반환: {"pairs": [(gi, di, sim)], "partial": {gi}, "gt_only": [gi], "dt_only": [di]}
    """
    gt_norm = [g["norm"] for g in gt_items]
    dt_norm = [d["norm"] for d in dt_items]
    g2d: dict[int, tuple[int, float]] = {}
    taken_d: set[int] = set()

    # (1) 순서를 보존하는 일치 블록 — 빈 문자열은 매칭 대상에서 제외한다.
    sm = SequenceMatcher(None, gt_norm, dt_norm, autojunk=False)
    for a, b, size in sm.get_matching_blocks():
        for k in range(size):
            if gt_norm[a + k]:
                g2d[a + k] = (b + k, 1.0)
                taken_d.add(b + k)

    # (2) 남은 요소끼리 유사도 그리디 배정 (순서 뒤바뀜·분할 문단을 여기서 회수)
    rem_g = [i for i in range(len(gt_items)) if i not in g2d and gt_norm[i]]
    rem_d = [j for j in range(len(dt_items)) if j not in taken_d and dt_norm[j]]
    cand = []
    for i in rem_g:
        for j in rem_d:
            thr = pair_threshold(gt_items[i], dt_items[j]) if pair_threshold else threshold
            s = similarity(gt_norm[i], dt_norm[j])
            if s >= thr:
                cand.append((s, i, j))
    cand.sort(key=lambda x: (-x[0], x[1], x[2]))
    used_g: set[int] = set()
    for s, i, j in cand:
        if i in used_g or j in taken_d:
            continue
        g2d[i] = (j, s)
        used_g.add(i)
        taken_d.add(j)

    # (3) 포함 관계 구제 — 한쪽 텍스트가 다른 쪽에 통째로 들어 있으면 같은 요소로 본다.
    #     "제2조 (정의)" vs "제2조 (정의) 개정 v1.1" 처럼 조판 배지가 섞여 들어온 경우가 여기다.
    #     찾긴 찾았는데 텍스트가 오염된 것이므로 '미탐지'가 아니라 partial 로 표시한다.
    partial: set[int] = set()
    if containment > 0:
        cand2 = []
        for i in (x for x in range(len(gt_items)) if x not in g2d and gt_norm[x]):
            for j in (y for y in range(len(dt_items)) if y not in taken_d and dt_norm[y]):
                a, b = gt_norm[i], dt_norm[j]
                short, long = (a, b) if len(a) <= len(b) else (b, a)
                cov = len(short) / len(long)
                if short in long and cov >= containment:
                    cand2.append((cov, i, j))
        cand2.sort(key=lambda x: (-x[0], x[1], x[2]))
        for cov, i, j in cand2:
            if i in g2d or j in taken_d:
                continue
            g2d[i] = (j, cov)
            taken_d.add(j)
            partial.add(i)

    pairs = sorted((i, j, s) for i, (j, s) in g2d.items())
    return {
        "pairs": pairs,
        "partial": partial,
        "gt_only": [i for i in range(len(gt_items)) if i not in g2d],
        "dt_only": [j for j in range(len(dt_items)) if j not in taken_d],
    }


def prf(tp: int, fp: int, fn: int) -> tuple[float, float, float]:
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f


ERROR_CASES: list[dict] = []      # §7에서 집계할 전 영역 공통 오류 버킷


def log_error(area: str, etype: str, doc: str, page, expected: str,
              detected: str, note: str = "") -> None:
    ERROR_CASES.append({
        "Area": area, "Error Type": etype, "Document": doc,
        "Page": page, "Expected": str(expected)[:160],
        "Detected": str(detected)[:160], "Note": note,
    })


def layout_pair_threshold(g: dict, d: dict) -> float:
    """표끼리는 셀 순서·분할 때문에 문자열 유사도가 낮아 임계값을 완화한다."""
    if g["Type"] == "Table" and d["Type"] == "Table":
        return TABLE_MATCH_THRESHOLD
    return MATCH_THRESHOLD


ALIGNMENTS = {n: align_sequences(GT_ELEMENTS[n], DT_ELEMENTS[n],
                                 pair_threshold=layout_pair_threshold,
                                 containment=CONTAINMENT)
              for n in scored_docs}
print("정렬 완료:", {n: f"{len(a['pairs'])}쌍(부분일치 {len(a['partial'])})"
                     for n, a in ALIGNMENTS.items()})

정렬 완료: {'법인카드_사용규정': '97쌍(부분일치 1)', '부서소개': '15쌍(부분일치 0)', '업무추진비_사용규정': '78쌍(부분일치 1)', '조직도': '22쌍(부분일치 0)', '조직설계_상세기획서': '59쌍(부분일치 0)', '직급체계': '18쌍(부분일치 0)', '출장비_사용규정': '63쌍(부분일치 2)', '회식_운영규정': '63쌍(부분일치 2)'}


### 3-1. Layout Detection — 요소 탐지율

GT 요소 대비 탐지/누락/오탐을 세고 Precision / Recall / F1을 낸다.

In [8]:
layout_rows = []
for name in scored_docs:
    gt, dt, al = GT_ELEMENTS[name], DT_ELEMENTS[name], ALIGNMENTS[name]
    tp, fn, fp = len(al["pairs"]), len(al["gt_only"]), len(al["dt_only"])
    p, r, f = prf(tp, fp, fn)
    layout_rows.append({
        "Document": name, "GT": len(gt), "Docling": len(dt),
        "Detected(TP)": tp, "Text Mismatch": len(al["partial"]),
        "Missing(FN)": fn, "Extra(FP)": fp,
        "Precision": round(p, 3), "Recall": round(r, 3), "F1": round(f, 3),
    })

    for i, j, _s in al["pairs"]:
        if i in al["partial"]:
            log_error("Layout", "Text Mismatch", name, dt[j]["Page"],
                      gt[i]["Text"], dt[j]["Text"], "부분 일치 — 텍스트 오염/분할")
    for i in al["gt_only"]:
        log_error("Layout", "Layout Missing", name, None,
                  f"[{gt[i]['Type']}] {gt[i]['Text']}", "(없음)", "GT 요소 미탐지")
    for j in al["dt_only"]:
        log_error("Layout", "Layout Extra", name, dt[j]["Page"], "(없음)",
                  f"[{dt[j]['Type']}] {dt[j]['Text']}", "GT에 없는 요소")

layout_detect_df = pd.DataFrame(layout_rows)
_tp = layout_detect_df["Detected(TP)"].sum()
_fn = layout_detect_df["Missing(FN)"].sum()
_fp = layout_detect_df["Extra(FP)"].sum()
LAYOUT_DETECT_P, LAYOUT_DETECT_R, LAYOUT_DETECT_F1 = prf(_tp, _fp, _fn)

display(layout_detect_df)
print(f"[전체] TP={_tp} FN={_fn} FP={_fp} → "
      f"P={LAYOUT_DETECT_P:.3f} R={LAYOUT_DETECT_R:.3f} F1={LAYOUT_DETECT_F1:.3f}")

_furn = layout_df[layout_df["Element Type"].isin(FURNITURE_TYPES) &
                  layout_df["Document"].isin(scored_docs)]
_pic = layout_df[layout_df["Element Type"].isin(UNVERIFIABLE_TYPES) &
                 layout_df["Document"].isin(scored_docs)]
print(f"참고: 대조 제외 Header/Footer {len(_furn)}건, GT 판정 불가 Picture {len(_pic)}건")

,Document,GT,Docling,Detected(TP),Text Mismatch,Missing(FN),Extra(FP),Precision,Recall,F1
0,법인카드_사용규정,98,102,97,1,1,5,0.951,0.990,0.970
1,부서소개,16,19,15,0,1,4,0.789,0.938,0.857
2,업무추진비_사용규정,80,86,78,1,2,8,0.907,0.975,0.940
3,조직도,22,26,22,0,0,4,0.846,1.000,0.917
4,조직설계_상세기획서,59,67,59,0,0,8,0.881,1.000,0.937
5,직급체계,19,21,18,0,1,3,0.857,0.947,0.900
6,출장비_사용규정,66,68,63,2,3,5,0.926,0.955,0.940
7,회식_운영규정,67,77,63,2,4,14,0.818,0.940,0.875


[전체] TP=415 FN=12 FP=51 → P=0.891 R=0.972 F1=0.929
참고: 대조 제외 Header/Footer 147건, GT 판정 불가 Picture 2건


### 3-2. Layout Element Classification — 요소 타입 분류

정렬에 성공한 쌍만 대상으로 **GT 타입 vs docling 타입** 혼동행렬과 타입별 P/R/F1을 낸다.
(탐지 자체를 실패한 요소는 §3-1에서 이미 셈했으므로 여기서 이중으로 벌하지 않는다.)

In [9]:
cls_pairs = []
for name in scored_docs:
    gt, dt = GT_ELEMENTS[name], DT_ELEMENTS[name]
    for i, j, s in ALIGNMENTS[name]["pairs"]:
        cls_pairs.append({
            "Document": name, "Page": dt[j]["Page"],
            "GT Type": gt[i]["Type"], "DT Type": dt[j]["Type"],
            "GT Text": gt[i]["Text"], "DT Text": dt[j]["Text"], "Sim": round(s, 3),
        })
        if gt[i]["Type"] != dt[j]["Type"]:
            log_error("Layout", "Layout Error", name, dt[j]["Page"],
                      gt[i]["Type"], f"{dt[j]['Type']} ({dt[j]['RawType']})",
                      gt[i]["Text"][:60])

cls_df = pd.DataFrame(cls_pairs)
if cls_df.empty:
    LAYOUT_TYPE_ACC = None
    print("분류 평가 대상 쌍이 없습니다 → N/A")
else:
    confusion = pd.crosstab(cls_df["GT Type"], cls_df["DT Type"],
                            margins=True, margins_name="ALL")
    LAYOUT_TYPE_ACC = float((cls_df["GT Type"] == cls_df["DT Type"]).mean())
    per_type = []
    for t in sorted(set(cls_df["GT Type"]) | set(cls_df["DT Type"])):
        tp = int(((cls_df["GT Type"] == t) & (cls_df["DT Type"] == t)).sum())
        fp = int(((cls_df["GT Type"] != t) & (cls_df["DT Type"] == t)).sum())
        fn = int(((cls_df["GT Type"] == t) & (cls_df["DT Type"] != t)).sum())
        p, r, f = prf(tp, fp, fn)
        per_type.append({"Type": t, "TP": tp, "FP": fp, "FN": fn,
                         "Precision": round(p, 3), "Recall": round(r, 3), "F1": round(f, 3)})
    layout_type_df = pd.DataFrame(per_type)

    print("[혼동행렬] 행=GT, 열=docling")
    display(confusion)
    print("[타입별 지표]")
    display(layout_type_df)
    print(f"타입 분류 정확도(정렬 쌍 기준) = {LAYOUT_TYPE_ACC:.3f}")

    _mis = cls_df[cls_df["GT Type"] != cls_df["DT Type"]]
    if not _mis.empty:
        print(f"\n오분류 {len(_mis)}건 중 상위 10건")
        display(_mis[["Document", "Page", "GT Type", "DT Type", "GT Text"]].head(10))

[혼동행렬] 행=GT, 열=docling


DT Type,Heading,List,Table,Text,ALL
GT Type,,,,,
Heading,114,0,0,4,118
List,0,183,0,0,183
Table,0,0,32,1,33
Text,20,7,0,54,81
ALL,134,190,32,59,415


[타입별 지표]


,Type,TP,FP,FN,Precision,Recall,F1
0,Heading,114,20,4,0.851,0.966,0.905
1,List,183,7,0,0.963,1.000,0.981
2,Table,32,0,1,1.000,0.970,0.985
3,Text,54,5,27,0.915,0.667,0.771


타입 분류 정확도(정렬 쌍 기준) = 0.923

오분류 32건 중 상위 10건


,Document,Page,GT Type,DT Type,GT Text
1,법인카드_사용규정,1,Text,Heading,**타이거 주식회사 (Tiger Inc.)**
87,법인카드_사용규정,6,Text,List,이 규정의 개정은 경영지원본부의 제안으로 대표이사의 승인을 받아 시행한다.
89,법인카드_사용규정,6,Text,List,이 규정은 2026년 8월 1일부터 시행한다.
90,법인카드_사용규정,7,Heading,Text,별표 1. 직책별 법인카드 사용 한도
92,법인카드_사용규정,7,Text,List,"※ 한도는 보임 중인 **직책** 기준으로 적용되며, 직급(사원~전무)과는 무관하다. 직책이 변경된 ..."
93,법인카드_사용규정,7,Text,List,"※ 경영지원본부는 본부장 직위가 없으므로, 산하 각 부서장(인사부·재무회계부·총무구매부·법무부·IT운..."
94,법인카드_사용규정,7,Text,List,※ 식대·기업업무추진비 지출은 제10조 제2항에 따라 직책과 무관하게 건당 30만원 초과 시 사전승인...
95,법인카드_사용규정,7,Text,List,"※ ""이사""가 부서장을 겸직하는 경우 부서장 한도를, 본부장 대행으로 보임된 경우 본부장 한도를 적용..."
96,법인카드_사용규정,7,Text,List,"※ ""전무""가 복수본부 총괄로 보임된 경우에도 본부장 한도를 적용하며, 발령 문서상 총괄 대상 본부가..."
113,업무추진비_사용규정,1,Text,Heading,**타이거 주식회사 (Tiger Inc.)**


### 3-3. Bounding Box — **IoU는 N/A**, 대신 기하 검사

GT 마크다운에는 좌표가 없다. **정답 bbox가 없으므로 IoU를 계산하지 않는다**(§채점 원칙 2).
억지 수치 대신 docling bbox 자체의 **기하학적 이상**만 자동 탐지해 사람 검수 시트로 남긴다.

| 검사 | 판정 |
|---|---|
| `Zero/Negative Area` | `r ≤ l` 또는 `t ≤ b` (docling bbox는 좌하단 원점 → `t > b`가 정상) |
| `Out of Page` | 박스가 페이지 물리 크기를 벗어남 |
| `Heavy Overlap` | 같은 페이지의 다른 요소와 IoU > 0.5 — 영역 분할 실패 징후 |

여기서 쓰는 IoU는 **정답 대비 정확도가 아니라 docling 요소끼리의 겹침**이라는 점에 주의.

In [10]:
def parse_bbox(v) -> tuple[float, float, float, float] | None:
    if not isinstance(v, str) or "," not in v:
        return None
    try:
        l, t, r, b = (float(x) for x in v.split(","))
    except ValueError:
        return None
    return l, t, r, b


def box_iou(a, b) -> float:
    al, at, ar, ab = a
    bl, bt, br, bb = b
    ix = max(0.0, min(ar, br) - max(al, bl))
    iy = max(0.0, min(at, bt) - max(ab, bb))
    inter = ix * iy
    area_a = max(0.0, ar - al) * max(0.0, at - ab)
    area_b = max(0.0, br - bl) * max(0.0, bt - bb)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


page_size = {}
if not pdf_info_df.empty:
    for _, r in pdf_info_df.iterrows():
        w, h = r["Size (pt)"].split("x")
        page_size[r["Document"]] = (float(w), float(h))

geom_rows = []
for name in scored_docs + na_docs:
    sub = layout_df[layout_df["Document"] == name]
    by_page: dict[int, list[tuple]] = {}
    for _, r in sub.iterrows():
        bbox = parse_bbox(r.get("BBox (l,t,r,b)"))
        page = int(r["Page"]) if pd.notna(r["Page"]) else None
        issues = []
        if bbox is None:
            issues.append("No BBox")
        else:
            l, t, rr, b = bbox
            if rr <= l or t <= b:
                issues.append("Zero/Negative Area")
            w, h = page_size.get(name, (None, None))
            if w and (l < -2 or rr > w + 2 or b < -2 or t > h + 2):
                issues.append("Out of Page")
            by_page.setdefault(page, []).append((bbox, r["Element Type"], r["Order"]))
        for issue in issues:
            geom_rows.append({"Document": name, "Page": page, "Order": r["Order"],
                              "Element Type": r["Element Type"], "Issue": issue,
                              "Text": str(r["Text"])[:80]})

    for page, boxes in by_page.items():
        for x in range(len(boxes)):
            for y in range(x + 1, len(boxes)):
                iou = box_iou(boxes[x][0], boxes[y][0])
                if iou > 0.5:
                    geom_rows.append({
                        "Document": name, "Page": page, "Order": boxes[x][2],
                        "Element Type": f"{boxes[x][1]} vs {boxes[y][1]}",
                        "Issue": "Heavy Overlap", "Text": f"IoU={iou:.2f}",
                    })

geom_df = pd.DataFrame(geom_rows)
BBOX_IOU_SCORE = None      # ← 정답 bbox 없음. 절대 임의 산출하지 않는다.

print("Bounding Box IoU (정답 대비) : N/A — GT에 좌표 없음")
if geom_df.empty:
    print("기하 이상치: 0건")
else:
    display(geom_df["Issue"].value_counts().rename_axis("Issue").reset_index(name="Count"))
    display(geom_df.head(10))
    for _, r in geom_df.iterrows():
        log_error("Layout", "BBox Geometry", r["Document"], r["Page"],
                  "정상 영역", r["Issue"], str(r["Text"])[:60])

Bounding Box IoU (정답 대비) : N/A — GT에 좌표 없음
기하 이상치: 0건


---
## 4. Text Hierarchy 평가

> **docling이 문서의 논리적 계층 구조를 제대로 복원하는가?**

네 가지를 따로 잰다.

| 지표 | 정의 |
|---|---|
| **Heading Detection** | GT 헤딩을 찾았는가 (P / R / F1) |
| **Heading Level** | 찾은 헤딩의 레벨이 맞는가 |
| **Parent-Child** | 직상위 헤딩(부모)이 GT와 같은가 |
| **Document Order** | 헤딩이 GT와 같은 순서로 나오는가 |

### 레벨 오프셋에 대한 주의

GT는 `#`=문서 제목, `##`=제N장, `###`=제N조 구조다. docling에는 **"문서 제목" 개념이 없어**
`제N장`을 H1, `제N조`를 H2로 내보내는 일이 흔하다. 이는 계층 **관계**가 깨진 것이 아니라
**전체가 한 칸 밀린 것**이므로, 두 값을 모두 보고한다.

- `Level Acc (strict)` — GT 레벨 숫자와 그대로 일치
- `Level Acc (offset 보정)` — 문서별 최빈 오프셋 δ를 뺀 뒤 일치 → **점수에는 이 값을 쓴다**

δ가 문서마다 다르거나 0이 아니면 리포트에 그대로 표기해 판단 근거를 남긴다.

같은 이유로 **부모-자식 비교에서는 GT의 문서 제목을 `<ROOT>`와 동치**로 본다. docling은 제목과
최상위 섹션을 구분하지 못해 GT에서 "제목의 자식"인 장(章)이 ROOT 직속으로 나오는데, 이는
레벨 지표가 이미 세고 있는 오프셋 문제라 부모 지표에서 다시 벌하면 같은 결함을 두 번 깎는다.
**"제N조가 제N장이 아니라 엉뚱한 헤딩 밑에 붙는"** 진짜 결함은 그대로 남는다.

In [11]:
def build_parent_map(items: list[dict]) -> dict[int, int | None]:
    """헤딩 시퀀스에서 각 헤딩의 부모(직상위 레벨 헤딩) 인덱스를 구한다."""
    parents: dict[int, int | None] = {}
    stack: list[tuple[int, int]] = []      # (level, idx)
    for idx, it in enumerate(items):
        lvl = it["Level"] or 1
        while stack and stack[-1][0] >= lvl:
            stack.pop()
        parents[idx] = stack[-1][1] if stack else None
        stack.append((lvl, idx))
    return parents


def lis_length(seq: list[int]) -> int:
    """최장 증가 부분수열 길이 — 순서 보존 비율 계산용."""
    import bisect
    tails: list[int] = []
    for x in seq:
        k = bisect.bisect_left(tails, x)
        if k == len(tails):
            tails.append(x)
        else:
            tails[k] = x
    return len(tails)


def doc_title_norm(gt_headings: list[dict]) -> str | None:
    """GT의 문서 제목(첫 최상위 헤딩) — 부모 비교에서 <ROOT>와 동치로 본다."""
    if not gt_headings:
        return None
    top = min((h["Level"] or 1) for h in gt_headings)
    first = gt_headings[0]
    return first["norm"] if (first["Level"] or 1) == top else None


def canon_parent(text: str, title_norm: str | None) -> str:
    """docling에는 '문서 제목' 개념이 없어 제목과 최상위 섹션이 같은 레벨로 나온다.

    그 탓에 GT에서 '제목의 자식'인 장(章)이 docling에서는 ROOT 직속이 되는데, 이는 레벨
    지표에서 이미 세는 오프셋 문제다. 부모 지표에서 이중으로 벌하지 않도록 제목=ROOT로 접는다.
    """
    return "<ROOT>" if title_norm and text == title_norm else text


hier_rows, hier_detail = [], []
HIER_TOTAL = {"tp": 0, "fp": 0, "fn": 0, "lvl_ok": 0, "lvl_ok_off": 0,
              "par_ok": 0, "matched": 0, "ord_ok": 0}

for name in scored_docs:
    gt_h = [g for g in GT_ELEMENTS[name] if g["Type"] == "Heading"]
    dt_h = [d for d in DT_ELEMENTS[name] if d["Type"] == "Heading"]
    al = align_sequences(gt_h, dt_h, containment=CONTAINMENT)
    tp, fn, fp = len(al["pairs"]), len(al["gt_only"]), len(al["dt_only"])
    p, r, f = prf(tp, fp, fn)

    gt_par, dt_par = build_parent_map(gt_h), build_parent_map(dt_h)
    title_norm = doc_title_norm(gt_h)
    g2d = {i: j for i, j, _ in al["pairs"]}

    deltas = [(dt_h[j]["Level"] or 1) - (gt_h[i]["Level"] or 1) for i, j in g2d.items()]
    offset = Counter(deltas).most_common(1)[0][0] if deltas else 0

    lvl_ok = lvl_ok_off = par_ok = 0
    for i, j in g2d.items():
        gl, dl = gt_h[i]["Level"] or 1, dt_h[j]["Level"] or 1
        if gl == dl:
            lvl_ok += 1
        if dl - offset == gl:
            lvl_ok_off += 1
        else:
            log_error("Hierarchy", "Heading Level Error", name, dt_h[j]["Page"],
                      f"H{gl}", f"H{dl}" + (f" (δ={offset:+d} 보정 후에도 불일치)" if offset else ""),
                      gt_h[i]["Text"][:60])

        gp, dp = gt_par[i], dt_par[j]
        gp_txt = canon_parent(gt_h[gp]["norm"] if gp is not None else "<ROOT>", title_norm)
        dp_txt = canon_parent(dt_h[dp]["norm"] if dp is not None else "<ROOT>", title_norm)
        same_parent = similarity(gp_txt, dp_txt) >= MATCH_THRESHOLD
        if same_parent:
            par_ok += 1
        else:
            log_error("Hierarchy", "Parent-Child Error", name, dt_h[j]["Page"],
                      f"부모={gt_h[gp]['Text'][:40] if gp is not None else 'ROOT'}",
                      f"부모={dt_h[dp]['Text'][:40] if dp is not None else 'ROOT'}",
                      gt_h[i]["Text"][:60])

    ordered = [g2d[i] for i in sorted(g2d)]
    keep = lis_length(ordered)
    order_acc = keep / len(ordered) if ordered else None
    for k in range(1, len(ordered)):
        if ordered[k] < ordered[k - 1]:
            gi = sorted(g2d)[k]
            log_error("Hierarchy", "Document Order Error", name, dt_h[ordered[k]]["Page"],
                      f"GT {k + 1}번째 헤딩", f"docling 순서 역전(Order {dt_h[ordered[k]]['Order']})",
                      gt_h[gi]["Text"][:60])

    for i in al["partial"]:
        log_error("Hierarchy", "Heading Text Mismatch", name, dt_h[g2d[i]]["Page"],
                  gt_h[i]["Text"], dt_h[g2d[i]]["Text"], "헤딩은 찾았으나 텍스트가 다름")
    for i in al["gt_only"]:
        log_error("Hierarchy", "Heading Missing", name, None,
                  f"H{gt_h[i]['Level']} {gt_h[i]['Text']}", "(없음)", "헤딩 미탐지")
    for j in al["dt_only"]:
        log_error("Hierarchy", "Heading Extra", name, dt_h[j]["Page"], "(없음)",
                  f"H{dt_h[j]['Level']} {dt_h[j]['Text']}", "GT에 없는 헤딩")

    hier_rows.append({
        "Document": name, "GT H": len(gt_h), "DT H": len(dt_h),
        "TP": tp, "Text Mismatch": len(al["partial"]), "Missing": fn, "Extra": fp,
        "Detect F1": round(f, 3),
        "Level δ": offset,
        "Level Acc(strict)": round(lvl_ok / tp, 3) if tp else None,
        "Level Acc(offset)": round(lvl_ok_off / tp, 3) if tp else None,
        "Parent Acc": round(par_ok / tp, 3) if tp else None,
        "Order Acc": round(order_acc, 3) if order_acc is not None else None,
        "GT Levels": "/".join(str(x) for x in sorted({g["Level"] for g in gt_h})),
        "DT Levels": "/".join(str(x) for x in sorted({d["Level"] or 1 for d in dt_h})),
    })
    HIER_TOTAL["tp"] += tp; HIER_TOTAL["fp"] += fp; HIER_TOTAL["fn"] += fn
    HIER_TOTAL["lvl_ok"] += lvl_ok; HIER_TOTAL["lvl_ok_off"] += lvl_ok_off
    HIER_TOTAL["par_ok"] += par_ok; HIER_TOTAL["matched"] += tp
    HIER_TOTAL["ord_ok"] += keep

hier_df = pd.DataFrame(hier_rows)
_m = HIER_TOTAL["matched"]
HIER_DETECT_P, HIER_DETECT_R, HIER_DETECT_F1 = prf(HIER_TOTAL["tp"], HIER_TOTAL["fp"], HIER_TOTAL["fn"])
HIER_LEVEL_ACC_STRICT = HIER_TOTAL["lvl_ok"] / _m if _m else None
HIER_LEVEL_ACC = HIER_TOTAL["lvl_ok_off"] / _m if _m else None
HIER_PARENT_ACC = HIER_TOTAL["par_ok"] / _m if _m else None
HIER_ORDER_ACC = HIER_TOTAL["ord_ok"] / _m if _m else None

display(hier_df)
print(f"[전체] Heading Detection P={HIER_DETECT_P:.3f} R={HIER_DETECT_R:.3f} F1={HIER_DETECT_F1:.3f}")
print(f"       Level Acc  strict={HIER_LEVEL_ACC_STRICT:.3f} / offset보정={HIER_LEVEL_ACC:.3f}")
print(f"       Parent-Child={HIER_PARENT_ACC:.3f}   Document Order={HIER_ORDER_ACC:.3f}")

,Document,GT H,DT H,TP,Text Mismatch,Missing,Extra,Detect F1,Level δ,Level Acc(strict),Level Acc(offset),Parent Acc,Order Acc,GT Levels,DT Levels
0,법인카드_사용규정,28,28,27,1,1,1,0.964,-1,0.037,0.963,0.667,0.926,1/2/3,1/2
1,부서소개,8,9,8,0,0,1,0.941,-1,0.125,0.875,1.000,1.000,1/2,1
2,업무추진비_사용규정,24,25,22,1,2,3,0.898,-1,0.045,0.955,0.818,0.955,1/2/3,1/2
3,조직도,5,9,4,0,1,5,0.571,-1,0.250,0.750,1.000,0.750,1/2,1
4,조직설계_상세기획서,12,23,10,0,2,13,0.571,0,1.000,1.000,0.800,1.000,1/2,1/2
5,직급체계,5,6,5,0,0,1,0.909,-1,0.200,0.800,1.000,1.000,1/2,1
6,출장비_사용규정,23,24,20,2,3,4,0.851,-1,0.050,0.950,0.500,0.850,1/2/3,1/2
7,회식_운영규정,20,21,18,2,2,3,0.878,-1,0.056,0.889,0.722,0.889,1/2/3,1/2


[전체] Heading Detection P=0.786 R=0.912 F1=0.844
       Level Acc  strict=0.149 / offset보정=0.930
       Parent-Child=0.737   Document Order=0.921


### 4-1. 계층 트리 대조 (샘플)

숫자만으로는 어디가 어긋났는지 안 보인다. 한 문서를 골라 **GT 트리와 docling 트리를 나란히**
찍는다. `PREVIEW_DOC`을 바꿔 다른 문서도 확인할 수 있다.

In [12]:
PREVIEW_DOC = scored_docs[0] if scored_docs else None


def render_heading_tree(items: list[dict], base: int = 1, limit: int = 40) -> list[str]:
    out = []
    for it in items[:limit]:
        lvl = it["Level"] or 1
        out.append("  " * max(0, lvl - base) + f"H{lvl} {it['Text'][:44]}")
    if len(items) > limit:
        out.append(f"... (+{len(items) - limit})")
    return out


if PREVIEW_DOC:
    gt_h = [g for g in GT_ELEMENTS[PREVIEW_DOC] if g["Type"] == "Heading"]
    dt_h = [d for d in DT_ELEMENTS[PREVIEW_DOC] if d["Type"] == "Heading"]
    left = render_heading_tree(gt_h)
    right = render_heading_tree(dt_h)
    width = max((len(x) for x in left), default=0) + 4
    print(f"=== {PREVIEW_DOC} ===")
    print(f"{'[GT]'.ljust(width)}[Docling]")
    print("-" * (width + 50))
    for a, b in zip(left + [""] * max(0, len(right) - len(left)),
                    right + [""] * max(0, len(left) - len(right))):
        print(f"{a.ljust(width)}{b}")

=== 법인카드_사용규정 ===
[GT]                            [Docling]
----------------------------------------------------------------------------------
H1 법인카드 사용 규정                   H1 타 이 거 주 식 회 사 ( T i ge r I n c . )
  H2 제1장 총칙                     H1 법인카드 사용 규정
    H3 제1조 (목적)                   H2 제1조 (목적)
    H3 제2조 (정의)                   H2 제2조 (정의) 개정 v1.1
    H3 제3조 (적용범위)                 H2 제3조 (적용범위)
  H2 제2장 법인카드의 발급 및 관리          H1 제1장 총칙
    H3 제4조 (발급 대상)                H2 제4조 (발급 대상)
    H3 제5조 (발급 절차)                H2 제5조 (발급 절차)
    H3 제6조 (관리 책임)                H2 제6조 (관리 책임)
    H3 제7조 (분실·도난 시 조치)           H2 제7조 (분실·도난 시 조치)
  H2 제3장 법인카드 사용 원칙             H1 제3장 법인카드 사용 원칙
    H3 제8조 (사용 가능 항목)             H2 제8조 (사용 가능 항목)
    H3 제9조 (사용 제한 및 금지 항목)      H1 제2장 법인카드의 발급 및 관리
    H3 제10조 (사용 한도)               H2 제9조 (사용 제한 및 금지 항목)
  H2 제4장 증빙 및 정산                  H2 제10조 (사용 한도) 개정 v1.1
    H3 제11조 (증빙서류의 원칙)          H1 제4장 증빙 및 정산
    H3 제12조 (정산 기한 및 절차)         

---
## 5. Table Structure 평가

> **docling이 표의 구조와 데이터를 제대로 복원하는가?**

docling 표는 `output/tables/<문서>/table_NN.json`의 **셀 격자**(`row`/`col`/`row_span`/`col_span`/
`column_header`/`text`)에서 읽는다(§1-3의 `load_docling_tables`). 마크다운 문자열이 아니라 격자를
쓰는 이유는, 마크다운으로 내보내는 순간 병합·헤더 정보가 사라지기 때문이다.

병합 셀은 앵커 셀 하나만 저장되므로 **span만큼 격자를 채워** 조밀 격자로 만든 뒤 비교한다.

GT 표와 docling 표의 대응은 순서가 아니라 **셀 텍스트 다중집합의 자카드 유사도**로 찾는다.
행·열이 밀리거나 표가 쪼개져도 같은 표를 알아보게 하기 위해서다.

In [13]:
def cell_bag(grid: list[list[str]]) -> Counter:
    return Counter(norm_loose(c) for row in grid for c in row if norm_loose(c))


def table_similarity(a: list[list[str]], b: list[list[str]]) -> float:
    """셀 텍스트 다중집합 자카드 — 행/열이 밀려도 대응 표를 찾아낸다."""
    ca, cb = cell_bag(a), cell_bag(b)
    if not ca or not cb:
        return 0.0
    inter = sum((ca & cb).values())
    union = sum((ca | cb).values())
    return inter / union if union else 0.0


print("docling 표(§1-3에서 로드):", {n: len(v) for n, v in DT_TABLES.items() if v})
print("GT 표                    :", {n: len(v) for n, v in GT_TABLES.items() if v})

docling 표(§1-3에서 로드): {'법인세법': 4, '법인카드_사용규정': 2, '부가가치세법': 3, '부서소개': 9, '업무추진비_사용규정': 2, '조직도': 4, '조직설계_상세기획서': 12, '직급체계': 3, '출장비_사용규정': 4, '회식_운영규정': 11}
GT 표                    : {'법인카드_사용규정': 2, '부서소개': 7, '업무추진비_사용규정': 3, '조직도': 2, '조직설계_상세기획서': 6, '직급체계': 2, '출장비_사용규정': 3, '회식_운영규정': 9}


### 5-1. Table Detection — 표 탐지

GT 표 하나에 docling 표가 **여러 개 대응하면 `Split`**(표가 페이지 경계 등에서 쪼개짐)으로 본다.
쪼개진 조각은 순서대로 이어 붙여 하나의 표로 놓고 구조를 비교한다 — 이렇게 하면 분할로 생긴
중복 헤더 행이 §5-2의 행 정확도에서 그대로 드러난다.

In [14]:
TABLE_MATCH: dict[str, list[dict]] = {}
det_rows = []

for name in scored_docs:
    gts, dts = GT_TABLES[name], DT_TABLES.get(name, [])
    assign: dict[int, list[int]] = {}          # gt idx -> [dt idx...]
    dt_unmatched = []
    for dj, dt in enumerate(dts):
        best_i, best_s = None, 0.0
        for gi, gt in enumerate(gts):
            s = table_similarity(gt["grid"], dt["grid"])
            if s > best_s:
                best_i, best_s = gi, s
        if best_i is not None and best_s >= TABLE_MATCH_THRESHOLD:
            assign.setdefault(best_i, []).append(dj)
        else:
            dt_unmatched.append(dj)

    tp = len(assign)
    fn = len(gts) - tp
    fp = len(dt_unmatched)
    splits = sum(1 for v in assign.values() if len(v) > 1)
    p, r, f = prf(tp, fp, fn)
    det_rows.append({"Document": name, "GT Tables": len(gts), "Detected Tables": len(dts),
                     "Matched": tp, "Missing": fn, "False Positive": fp, "Split": splits,
                     "Precision": round(p, 3), "Recall": round(r, 3), "F1": round(f, 3)})

    pairs = []
    for gi, gt in enumerate(gts):
        frags = assign.get(gi, [])
        if not frags:
            log_error("Table", "Table Detection Error", name, None,
                      f"GT 표 {gi + 1} ({gt['Rows']}x{gt['Cols']})", "(미탐지)", "표 누락")
            continue
        frags = sorted(frags)
        merged_grid = [row for dj in frags for row in dts[dj]["grid"]]
        if len(frags) > 1:
            log_error("Table", "Table Detection Error", name, dts[frags[0]]["Page"],
                      f"GT 표 {gi + 1} 1개", f"docling 표 {len(frags)}개로 분할",
                      "표 분할(Split)")
        pairs.append({"gt_idx": gi, "dt_idx": frags,
                      "gt": gt, "dt_first": dts[frags[0]],
                      "merged_grid": merged_grid,
                      "merged_header": dts[frags[0]]["HeaderRows"],
                      "header_detected": dts[frags[0]]["HeaderDetected"],
                      "merged_count": sum(dts[dj]["Merged"] for dj in frags)})
    for dj in dt_unmatched:
        log_error("Table", "Table Detection Error", name, dts[dj]["Page"], "(없음)",
                  f"docling 표 {dts[dj]['Table']} ({dts[dj]['Rows']}x{dts[dj]['Cols']})",
                  "GT에 없는 표(오탐)")
    TABLE_MATCH[name] = pairs

table_detect_df = pd.DataFrame(det_rows)
_tp = table_detect_df["Matched"].sum()
_fn = table_detect_df["Missing"].sum()
_fp = table_detect_df["False Positive"].sum()
TABLE_DETECT_P, TABLE_DETECT_R, TABLE_DETECT_F1 = prf(_tp, _fp, _fn)
display(table_detect_df)
print(f"[전체] 매칭={_tp} 누락={_fn} 오탐={_fp} 분할={table_detect_df['Split'].sum()} → "
      f"P={TABLE_DETECT_P:.3f} R={TABLE_DETECT_R:.3f} F1={TABLE_DETECT_F1:.3f}")

,Document,GT Tables,Detected Tables,Matched,Missing,False Positive,Split,Precision,Recall,F1
0,법인카드_사용규정,2,2,2,0,0,0,1.000,1.000,1.000
1,부서소개,7,9,7,0,0,2,1.000,1.000,1.000
2,업무추진비_사용규정,3,2,2,1,0,0,1.000,0.667,0.800
3,조직도,2,4,2,0,1,1,0.667,1.000,0.800
4,조직설계_상세기획서,6,12,6,0,1,4,0.857,1.000,0.923
5,직급체계,2,3,2,0,0,1,1.000,1.000,1.000
6,출장비_사용규정,3,4,3,0,0,1,1.000,1.000,1.000
7,회식_운영규정,9,11,8,1,1,2,0.889,0.889,0.889


[전체] 매칭=32 누락=2 오탐=3 분할=11 → P=0.914 R=0.941 F1=0.928


### 5-2. Table Structure Accuracy — 행 / 열 / 헤더

매칭된 표 쌍마다 행 수·열 수·헤더 인식을 비교한다. 셀 값 비교는 §5-3.

- **Row / Column Accuracy** — 행·열 수가 정확히 같은 표의 비율 (여유를 주지 않는다)
- **Header Accuracy** — docling이 헤더 행을 인식했고, 그 텍스트가 GT 첫 행과 일치하는가

In [15]:
struct_rows = []
for name in scored_docs:
    for pr in TABLE_MATCH[name]:
        gt, grid = pr["gt"], pr["merged_grid"]
        dt_rows_n, dt_cols_n = len(grid), max((len(r) for r in grid), default=0)
        row_ok = dt_rows_n == gt["Rows"]
        col_ok = dt_cols_n == gt["Cols"]

        gt_header = [norm_loose(c) for c in gt["grid"][0]] if gt["HeaderRows"] else []
        dt_header = [norm_loose(c) for c in grid[0]] if grid else []
        header_ok = bool(pr["header_detected"]) and gt_header == dt_header

        struct_rows.append({
            "Document": name, "GT Table": pr["gt_idx"] + 1,
            "DT Table": ",".join(str(dt + 1) for dt in pr["dt_idx"]),
            "GT RxC": f"{gt['Rows']}x{gt['Cols']}", "DT RxC": f"{dt_rows_n}x{dt_cols_n}",
            "Row OK": row_ok, "Col OK": col_ok,
            "Header Detected": pr["header_detected"], "Header OK": header_ok,
        })
        if not row_ok:
            log_error("Table", "Row/Column Error", name, pr["dt_first"]["Page"],
                      f"Rows {gt['Rows']}", f"Rows {dt_rows_n}", f"GT 표 {pr['gt_idx'] + 1}")
        if not col_ok:
            log_error("Table", "Row/Column Error", name, pr["dt_first"]["Page"],
                      f"Cols {gt['Cols']}", f"Cols {dt_cols_n}", f"GT 표 {pr['gt_idx'] + 1}")
        if not header_ok:
            log_error("Table", "Header Error", name, pr["dt_first"]["Page"],
                      " | ".join(c for c in gt["grid"][0]) if gt["HeaderRows"] else "(헤더 없음)",
                      (" | ".join(grid[0]) if pr["header_detected"] else "헤더 미인식"),
                      f"GT 표 {pr['gt_idx'] + 1}")

table_struct_df = pd.DataFrame(struct_rows)
if table_struct_df.empty:
    TABLE_ROW_ACC = TABLE_COL_ACC = TABLE_HEADER_ACC = None
    print("매칭된 표가 없습니다 → 구조 지표 N/A")
else:
    TABLE_ROW_ACC = float(table_struct_df["Row OK"].mean())
    TABLE_COL_ACC = float(table_struct_df["Col OK"].mean())
    TABLE_HEADER_ACC = float(table_struct_df["Header OK"].mean())
    display(table_struct_df)
    print(f"Row Accuracy={TABLE_ROW_ACC:.3f}  Column Accuracy={TABLE_COL_ACC:.3f}  "
          f"Header Accuracy={TABLE_HEADER_ACC:.3f}  (표 {len(table_struct_df)}개 기준)")

,Document,GT Table,DT Table,GT RxC,DT RxC,Row OK,Col OK,Header Detected,Header OK
0,법인카드_사용규정,1,1,6x2,3x2,False,True,False,False
1,법인카드_사용규정,2,2,6x4,6x4,True,True,True,True
2,부서소개,1,1,6x5,6x5,True,True,True,True
3,부서소개,2,2,3x5,3x5,True,True,True,True
4,부서소개,3,3,3x5,3x5,True,True,True,True
5,부서소개,4,4,4x5,4x5,True,True,True,True
6,부서소개,5,"5,6",4x5,5x5,False,True,True,True
7,부서소개,6,7,2x5,2x5,True,True,True,True
8,부서소개,7,"8,9",12x3,13x3,False,True,True,True
9,업무추진비_사용규정,2,1,4x3,4x3,True,True,True,True


Row Accuracy=0.594  Column Accuracy=1.000  Header Accuracy=0.938  (표 32개 기준)


### 5-3. Table Content Accuracy — 셀 값

행 수가 어긋난 표에서 위치 비교만 하면 한 칸 밀린 전체가 오답이 되어 실제보다 과하게 나쁘다.
그래서 **행을 먼저 정렬**(§3-0의 같은 정렬 엔진, 행 전체 텍스트를 키로)한 뒤, 정렬된 행 안에서
열 순서대로 셀을 비교한다. 정렬되지 않은 GT 행의 셀은 전부 오답으로 센다.

| 지표 | 판정 |
|---|---|
| `Cell Acc (loose)` | 정규화 후 값 일치 — **점수에 쓰는 값** |
| `Cell Acc (strict)` | 공백까지 그대로 일치 |

두 값의 차이가 곧 **조판 유래 공백·자간 결함의 크기**다. 오류는 아래로 세분한다.

`공백·자간` / `숫자·금액 오류` / `문자 누락` / `문자 추가` / `빈 셀` / `내용 불일치`

In [16]:
_DIGITS = re.compile(r"\d")


def classify_cell_error(gt_text: str, dt_text: str) -> str:
    g_s, d_s = norm_strict(gt_text), norm_strict(dt_text)
    g_l, d_l = norm_loose(gt_text), norm_loose(dt_text)
    if g_l == d_l:
        return "공백·자간" if g_s != d_s else "일치"
    if not d_l:
        return "빈 셀"
    g_num = "".join(_DIGITS.findall(g_s))
    d_num = "".join(_DIGITS.findall(d_s))
    if (g_num or d_num) and g_num != d_num:
        return "숫자·금액 오류"
    if g_l in d_l:
        return "문자 추가"
    if d_l in g_l:
        return "문자 누락"
    return "내용 불일치"


cell_stat = Counter()
cell_rows, cell_errors = [], []

for name in scored_docs:
    for pr in TABLE_MATCH[name]:
        gt, grid = pr["gt"], pr["merged_grid"]
        gt_sig = [{"norm": norm_loose(" ".join(r)), "row": r} for r in gt["grid"]]
        dt_sig = [{"norm": norm_loose(" ".join(r)), "row": r} for r in grid]
        al = align_sequences(gt_sig, dt_sig, threshold=0.60)
        row_map = {i: j for i, j, _ in al["pairs"]}

        total = ok_loose = ok_strict = 0
        for gi, g_row in enumerate(gt["grid"]):
            d_row = grid[row_map[gi]] if gi in row_map else None
            for ci, g_cell in enumerate(g_row):
                total += 1
                d_cell = d_row[ci] if d_row is not None and ci < len(d_row) else ""
                kind = classify_cell_error(g_cell, d_cell)
                if kind == "일치":
                    ok_loose += 1
                    ok_strict += 1
                    continue
                if kind == "공백·자간":
                    ok_loose += 1
                cell_stat[kind] += 1
                cell_errors.append({
                    "Document": name, "Table": pr["gt_idx"] + 1,
                    "Row": gi, "Col": ci, "Kind": kind,
                    "Expected": g_cell, "Detected": d_cell,
                    "Row Aligned": gi in row_map,
                })
        cell_rows.append({
            "Document": name, "GT Table": pr["gt_idx"] + 1, "GT Cells": total,
            "Rows Aligned": f"{len(row_map)}/{gt['Rows']}",
            "Cell Acc(loose)": round(ok_loose / total, 3) if total else None,
            "Cell Acc(strict)": round(ok_strict / total, 3) if total else None,
        })

table_cell_df = pd.DataFrame(cell_rows)
cell_error_df = pd.DataFrame(cell_errors)

if table_cell_df.empty:
    TABLE_CELL_ACC = TABLE_CELL_ACC_STRICT = None
    print("셀 비교 대상 없음 → N/A")
else:
    _tot = sum(r["GT Cells"] for r in cell_rows)
    _ok_l = sum(r["GT Cells"] * (r["Cell Acc(loose)"] or 0) for r in cell_rows)
    _ok_s = sum(r["GT Cells"] * (r["Cell Acc(strict)"] or 0) for r in cell_rows)
    TABLE_CELL_ACC = _ok_l / _tot
    TABLE_CELL_ACC_STRICT = _ok_s / _tot
    display(table_cell_df)
    print(f"[전체] GT 셀 {_tot}개 → Cell Acc(loose)={TABLE_CELL_ACC:.3f} / "
          f"(strict)={TABLE_CELL_ACC_STRICT:.3f}")
    if cell_stat:
        display(pd.DataFrame(sorted(cell_stat.items(), key=lambda x: -x[1]),
                             columns=["Cell Error Kind", "Count"]))
    if not cell_error_df.empty:
        strict_only = cell_error_df[cell_error_df["Kind"] != "공백·자간"]
        print(f"\n값이 실제로 틀린 셀 {len(strict_only)}건 중 상위 12건")
        display(strict_only[["Document", "Table", "Row", "Col", "Kind",
                             "Expected", "Detected"]].head(12))

if not cell_error_df.empty:
    for _, r in cell_error_df.iterrows():
        log_error("Table",
                  "OCR/Text Error" if r["Kind"] == "공백·자간" else "Cell Content Error",
                  r["Document"], None, r["Expected"], r["Detected"],
                  f"표 {r['Table']} ({r['Row']},{r['Col']}) — {r['Kind']}")

,Document,GT Table,GT Cells,Rows Aligned,Cell Acc(loose),Cell Acc(strict)
0,법인카드_사용규정,1,12,3/6,0.500,0.500
1,법인카드_사용규정,2,24,6/6,1.000,1.000
2,부서소개,1,30,6/6,1.000,0.533
3,부서소개,2,15,3/3,1.000,0.667
4,부서소개,3,15,3/3,1.000,0.533
5,부서소개,4,20,4/4,1.000,0.500
6,부서소개,5,20,4/4,1.000,0.500
7,부서소개,6,10,2/2,1.000,0.700
8,부서소개,7,36,12/12,1.000,0.889
9,업무추진비_사용규정,2,12,4/4,1.000,1.000


[전체] GT 셀 768개 → Cell Acc(loose)=0.965 / (strict)=0.755


,Cell Error Kind,Count
0,공백·자간,161
1,빈 셀,27



값이 실제로 틀린 셀 27건 중 상위 12건


,Document,Table,Row,Col,Kind,Expected,Detected
0,법인카드_사용규정,1,0,0,빈 셀,구분,
1,법인카드_사용규정,1,0,1,빈 셀,내용,
2,법인카드_사용규정,1,1,0,빈 셀,문서번호,
3,법인카드_사용규정,1,1,1,빈 셀,TIGER-REG-2026-003,
4,법인카드_사용규정,1,5,0,빈 셀,개정이력,
5,법인카드_사용규정,1,5,1,빈 셀,"v1.0 제정 (조문 정합성 검토 반영) / v1.1 개정: 제2조 관리자 정의 정비, 제10조 사전...",
59,조직도,2,1,0,빈 셀,임직원 수,
60,조직도,2,1,1,빈 셀,"약 1,050명",
108,조직설계_상세기획서,5,1,0,빈 셀,AI사업본부,
109,조직설계_상세기획서,5,1,1,빈 셀,플랫폼사업본부(클라우드부),


### 5-4. Merged Cell — **N/A** (+ 휴리스틱 참고)

GT가 마크다운 파이프 표라 **rowspan/colspan을 표현할 방법이 없다.** 따라서 병합 정확도는
`N/A`이며 점수에 넣지 않는다.

다만 원본에서 세로 병합된 칸은 마크다운으로 옮길 때 **같은 값이 연속 행에 반복**되는 형태로
남는 경우가 많다. 그 반복 패턴을 세어 docling이 보고한 병합 수와 나란히 보여준다.
**이것은 추정치이며 채점 근거가 아니다.**

In [17]:
def estimate_gt_merges(grid: list[list[str]]) -> int:
    """세로 방향으로 같은 값이 연속 반복되는 칸 수 — 병합의 마크다운 흔적(추정)."""
    if not grid:
        return 0
    est = 0
    ncols = max(len(r) for r in grid)
    for c in range(ncols):
        prev = None
        for r in range(len(grid)):
            v = norm_loose(grid[r][c]) if c < len(grid[r]) else ""
            if v and v == prev:
                est += 1
            prev = v
    return est


merge_rows = []
for name in scored_docs:
    for pr in TABLE_MATCH[name]:
        est = estimate_gt_merges(pr["gt"]["grid"])
        got = pr["merged_count"]
        merge_rows.append({"Document": name, "GT Table": pr["gt_idx"] + 1,
                           "GT 반복값(추정 병합)": est, "Docling Merged Cells": got,
                           "일치?": "-" if est == 0 and got == 0 else ("O" if est == got else "X")})

TABLE_MERGED_ACC = None      # ← GT 표현 불가. 임의 산출하지 않는다.
merged_df = pd.DataFrame(merge_rows)
print("Merged Cell Accuracy : N/A — 마크다운 GT는 병합을 표현할 수 없음 (점수 제외)")
if not merged_df.empty:
    display(merged_df[merged_df["일치?"] != "-"])

Merged Cell Accuracy : N/A — 마크다운 GT는 병합을 표현할 수 없음 (점수 제외)


,Document,GT Table,GT 반복값(추정 병합),Docling Merged Cells,일치?
8,부서소개,7,2,2,O
15,조직설계_상세기획서,3,3,1,X
16,조직설계_상세기획서,4,1,0,X
17,조직설계_상세기획서,5,2,1,X
19,직급체계,1,3,1,X
20,직급체계,2,1,0,X
22,출장비_사용규정,2,2,0,X
23,출장비_사용규정,3,1,0,X
24,회식_운영규정,2,2,2,O
27,회식_운영규정,5,2,0,X


### 5-5. GT 없는 문서 — 사람 검수 시트

법령 PDF 3종은 정답지가 없다. **점수를 만들지 않고**, 요소·표 통계와 기하 이상치를
`layout_manual_review.csv`로 떨궈 사람이 직접 보게 한다.

In [18]:
manual_rows = []
for name in na_docs:
    sub = layout_df[layout_df["Document"] == name]
    counts = sub["Element Type"].value_counts().to_dict()
    tbls = DT_TABLES.get(name, [])
    zero_dim = sum(1 for t in tbls if t["Rows"] == 0 or t["Cols"] == 0)
    issues = len(geom_df[geom_df["Document"] == name]) if not geom_df.empty else 0
    manual_rows.append({
        "Document": name, "GT": "없음", "Pages": int(sub["Page"].max()),
        "Elements": len(sub),
        **{k: counts.get(k, 0) for k in ["Heading", "Text", "List", "Table", "Picture",
                                         "Header", "Footer"]},
        "Tables(JSON)": len(tbls), "Tables 0x0": zero_dim, "BBox Issues": issues,
        "Verdict(사람 입력)": "", "Comment(사람 입력)": "",
    })

manual_df = pd.DataFrame(manual_rows)
if not manual_df.empty:
    manual_df.to_csv(EVAL_DIR / "layout_manual_review.csv", index=False, encoding="utf-8-sig")
    print(f"사람 검수 대상 {len(manual_df)}종 → layout_manual_review.csv")
    display(manual_df)
else:
    print("GT 없는 문서 없음")

사람 검수 대상 3종 → layout_manual_review.csv


,Document,GT,Pages,Elements,Heading,Text,List,Table,Picture,Header,Footer,Tables(JSON),Tables 0x0,BBox Issues,Verdict(사람 입력),Comment(사람 입력)
0,법인세법,없음,97,2098,188,387,1216,4,3,97,194,4,4,0,,
1,부가가치세법,없음,35,767,57,105,493,3,1,35,70,3,3,0,,
2,여신전문금융업법,없음,34,908,103,167,530,0,1,34,68,0,0,0,,


---
## 6. 정량 평가 점수 산출

영역별 100점 만점, 최종 점수는 다음과 같이 가중 합산한다.

```text
Overall Score = Layout × 0.30 + Text Hierarchy × 0.30 + Table Structure × 0.40
```

표에 0.40을 주는 이유는, RAG에서 표 구조가 깨질 때 손실이 가장 크기 때문이다(행이 뒤섞인
한도표 한 장이 룰 판정 근거를 통째로 무너뜨린다).

### N/A 처리 규칙

`N/A`인 지표(정답 bbox 없음, 병합 셀 표현 불가)는 **0점으로 치지 않고 계산에서 제외**하며,
그 가중치는 같은 영역의 남은 지표에 비례 배분한다. 어떤 지표가 빠졌고 가중치가 어떻게
재분배됐는지는 아래 표의 `Status` / `Eff.Weight` 열에 그대로 남는다.

In [19]:
METRIC_SPEC = [
    # (영역, 지표, 값, 명목 가중치, 비고)
    ("Layout",    "Element Detection F1",      LAYOUT_DETECT_F1,      0.40, ""),
    ("Layout",    "Element Type Accuracy",     LAYOUT_TYPE_ACC,       0.35, "정렬된 쌍 기준"),
    ("Layout",    "Bounding Box IoU",          BBOX_IOU_SCORE,        0.25, "GT 좌표 없음"),
    ("Hierarchy", "Heading Detection F1",      HIER_DETECT_F1,        0.40, ""),
    ("Hierarchy", "Heading Level Accuracy",    HIER_LEVEL_ACC,        0.25, "문서별 오프셋 보정"),
    ("Hierarchy", "Parent-Child Accuracy",     HIER_PARENT_ACC,       0.20, ""),
    ("Hierarchy", "Document Order Accuracy",   HIER_ORDER_ACC,        0.15, ""),
    ("Table",     "Table Detection F1",        TABLE_DETECT_F1,       0.20, ""),
    ("Table",     "Row Accuracy",              TABLE_ROW_ACC,         0.15, "행 수 완전일치 비율"),
    ("Table",     "Column Accuracy",           TABLE_COL_ACC,         0.15, "열 수 완전일치 비율"),
    ("Table",     "Header Accuracy",           TABLE_HEADER_ACC,      0.15, ""),
    ("Table",     "Cell Accuracy",             TABLE_CELL_ACC,        0.25, "정규화 후 값 일치"),
    ("Table",     "Merged Cell Accuracy",      TABLE_MERGED_ACC,      0.10, "마크다운 GT 표현 불가"),
]

# 참고 지표 — 점수에는 넣지 않고 리포트에만 남긴다.
REFERENCE_METRICS = [
    ("Layout",    "Element Detection Precision", LAYOUT_DETECT_P),
    ("Layout",    "Element Detection Recall",    LAYOUT_DETECT_R),
    ("Hierarchy", "Heading Level Acc (strict)",  HIER_LEVEL_ACC_STRICT),
    ("Table",     "Cell Accuracy (strict)",      TABLE_CELL_ACC_STRICT),
]

rows = []
for area, metric, value, weight, note in METRIC_SPEC:
    avail = value is not None
    rows.append({"Category": area, "Metric": metric,
                 "Score": round(float(value), 3) if avail else None,
                 "Weight": weight, "Status": "OK" if avail else "N/A", "Note": note})
summary_df = pd.DataFrame(rows)

AREA_SCORES: dict[str, float | None] = {}
eff_weights = []
for area in ("Layout", "Hierarchy", "Table"):
    part = summary_df[summary_df["Category"] == area]
    ok = part[part["Status"] == "OK"]
    wsum = ok["Weight"].sum()
    for _, r in part.iterrows():
        eff_weights.append(round(r["Weight"] / wsum, 3) if r["Status"] == "OK" and wsum else 0.0)
    AREA_SCORES[area] = float((ok["Score"] * ok["Weight"]).sum() / wsum * 100) if wsum else None
summary_df["Eff.Weight"] = eff_weights

_ok_areas = {a: s for a, s in AREA_SCORES.items() if s is not None}
_wsum = sum(AREA_WEIGHTS[a] for a in _ok_areas)
OVERALL_SCORE = sum(s * AREA_WEIGHTS[a] for a, s in _ok_areas.items()) / _wsum if _wsum else None

summary_df = pd.concat([
    summary_df,
    pd.DataFrame([{"Category": a, "Metric": f"── {a} Score (100점)",
                   "Score": round(s, 1) if s is not None else None,
                   "Weight": AREA_WEIGHTS[a],
                   "Eff.Weight": round(AREA_WEIGHTS[a] / _wsum, 3) if _wsum and s is not None else 0.0,
                   "Status": "OK" if s is not None else "N/A", "Note": "영역 합산"}
                  for a, s in AREA_SCORES.items()]),
], ignore_index=True)

display(summary_df)
_na = summary_df[summary_df["Status"] == "N/A"]["Metric"].tolist()
print(f"N/A로 제외된 지표({len(_na)}): {', '.join(_na) if _na else '없음'}")
print("→ 위 지표의 가중치는 같은 영역의 남은 지표로 재분배됨(Eff.Weight 열).")

,Category,Metric,Score,Weight,Status,Note,Eff.Weight
0,Layout,Element Detection F1,0.929,0.40,OK,,0.533
1,Layout,Element Type Accuracy,0.923,0.35,OK,정렬된 쌍 기준,0.467
2,Layout,Bounding Box IoU,NaN,0.25,N/A,GT 좌표 없음,0.000
3,Hierarchy,Heading Detection F1,0.844,0.40,OK,,0.400
4,Hierarchy,Heading Level Accuracy,0.930,0.25,OK,문서별 오프셋 보정,0.250
5,Hierarchy,Parent-Child Accuracy,0.737,0.20,OK,,0.200
6,Hierarchy,Document Order Accuracy,0.921,0.15,OK,,0.150
7,Table,Table Detection F1,0.928,0.20,OK,,0.222
8,Table,Row Accuracy,0.594,0.15,OK,행 수 완전일치 비율,0.167
9,Table,Column Accuracy,1.000,0.15,OK,열 수 완전일치 비율,0.167


N/A로 제외된 지표(2): Bounding Box IoU, Merged Cell Accuracy
→ 위 지표의 가중치는 같은 영역의 남은 지표로 재분배됨(Eff.Weight 열).


### 6-1. 평가 결과 DataFrame (Category / Metric / Score)

In [20]:
metric_df = summary_df[~summary_df["Metric"].str.startswith("──")][
    ["Category", "Metric", "Score", "Status"]
].reset_index(drop=True)
ref_df = pd.DataFrame(
    [{"Category": a, "Metric": m, "Score": round(v, 3) if v is not None else None,
      "Status": "참고"} for a, m, v in REFERENCE_METRICS]
)
display(pd.concat([metric_df, ref_df], ignore_index=True))

,Category,Metric,Score,Status
0,Layout,Element Detection F1,0.929,OK
1,Layout,Element Type Accuracy,0.923,OK
2,Layout,Bounding Box IoU,NaN,N/A
3,Hierarchy,Heading Detection F1,0.844,OK
4,Hierarchy,Heading Level Accuracy,0.930,OK
5,Hierarchy,Parent-Child Accuracy,0.737,OK
6,Hierarchy,Document Order Accuracy,0.921,OK
7,Table,Table Detection F1,0.928,OK
8,Table,Row Accuracy,0.594,OK
9,Table,Column Accuracy,1.000,OK


### 6-2. 문서별 상세 (evaluation_details.csv)

In [21]:
detail_rows = []


def _push(frame: pd.DataFrame, category: str, cols: dict[str, str]) -> None:
    if frame is None or frame.empty:
        return
    for _, r in frame.iterrows():
        for col, metric in cols.items():
            if col in frame.columns and pd.notna(r[col]):
                detail_rows.append({
                    "Document": r["Document"], "Category": category, "Metric": metric,
                    "Value": r[col],
                    "Scope": f"table {r['GT Table']}" if "GT Table" in frame.columns else "document",
                })


_push(layout_detect_df, "Layout",
      {"Precision": "Element Detection Precision", "Recall": "Element Detection Recall",
       "F1": "Element Detection F1", "Missing(FN)": "Missing Elements",
       "Extra(FP)": "Extra Elements"})
_push(hier_df, "Hierarchy",
      {"Detect F1": "Heading Detection F1", "Level Acc(strict)": "Heading Level Acc (strict)",
       "Level Acc(offset)": "Heading Level Acc (offset)", "Parent Acc": "Parent-Child Acc",
       "Order Acc": "Document Order Acc", "Level δ": "Level Offset"})
_push(table_detect_df, "Table",
      {"F1": "Table Detection F1", "Missing": "Missing Tables",
       "False Positive": "False Positive Tables", "Split": "Split Tables"})
_push(table_cell_df, "Table",
      {"Cell Acc(loose)": "Cell Accuracy", "Cell Acc(strict)": "Cell Accuracy (strict)"})

details_df = pd.DataFrame(detail_rows)
display(details_df.head(25))
print(f"총 {len(details_df)}행 — 전체는 evaluation_details.csv 로 저장된다.")

,Document,Category,Metric,Value,Scope
0,법인카드_사용규정,Layout,Element Detection Precision,0.951,document
1,법인카드_사용규정,Layout,Element Detection Recall,0.990,document
2,법인카드_사용규정,Layout,Element Detection F1,0.970,document
3,법인카드_사용규정,Layout,Missing Elements,1.000,document
4,법인카드_사용규정,Layout,Extra Elements,5.000,document
5,부서소개,Layout,Element Detection Precision,0.789,document
6,부서소개,Layout,Element Detection Recall,0.938,document
7,부서소개,Layout,Element Detection F1,0.857,document
8,부서소개,Layout,Missing Elements,1.000,document
9,부서소개,Layout,Extra Elements,4.000,document


총 184행 — 전체는 evaluation_details.csv 로 저장된다.


### 6-3. 종합 결과 출력

In [22]:
def fmt(v, digits: int = 2) -> str:
    return "N/A" if v is None else f"{v:.{digits}f}"


lines = []
lines.append("=" * 50)
lines.append("        DOCLING PARSING EVALUATION")
lines.append("=" * 50)
lines.append(f"Documents scored : {len(scored_docs)}  (GT: tiger_inc/md)")
lines.append(f"Documents N/A    : {len(na_docs)}  (no ground truth)")
lines.append("")
lines.append("[1] Layout Analysis")
lines.append("-" * 50)
lines.append(f"Element Detection F1  : {fmt(LAYOUT_DETECT_F1)}")
lines.append(f"Element Type Accuracy : {fmt(LAYOUT_TYPE_ACC)}")
lines.append(f"Bounding Box IoU      : {fmt(BBOX_IOU_SCORE)}  (no GT coordinates)")
lines.append(f"Layout Score          : {fmt(AREA_SCORES['Layout'], 1)} / 100")
lines.append("")
lines.append("[2] Text Hierarchy")
lines.append("-" * 50)
lines.append(f"Heading Detection F1  : {fmt(HIER_DETECT_F1)}")
lines.append(f"Heading Level (offset): {fmt(HIER_LEVEL_ACC)}   (strict: {fmt(HIER_LEVEL_ACC_STRICT)})")
lines.append(f"Parent-Child Relation : {fmt(HIER_PARENT_ACC)}")
lines.append(f"Document Order        : {fmt(HIER_ORDER_ACC)}")
lines.append(f"Hierarchy Score       : {fmt(AREA_SCORES['Hierarchy'], 1)} / 100")
lines.append("")
lines.append("[3] Table Structure")
lines.append("-" * 50)
lines.append(f"Table Detection F1    : {fmt(TABLE_DETECT_F1)}")
lines.append(f"Row Accuracy          : {fmt(TABLE_ROW_ACC)}")
lines.append(f"Column Accuracy       : {fmt(TABLE_COL_ACC)}")
lines.append(f"Header Accuracy       : {fmt(TABLE_HEADER_ACC)}")
lines.append(f"Cell Accuracy         : {fmt(TABLE_CELL_ACC)}   (strict: {fmt(TABLE_CELL_ACC_STRICT)})")
lines.append(f"Merged Cell Accuracy  : {fmt(TABLE_MERGED_ACC)}  (not expressible in MD GT)")
lines.append(f"Table Score           : {fmt(AREA_SCORES['Table'], 1)} / 100")
lines.append("")
lines.append("=" * 50)
lines.append(f"OVERALL SCORE : {fmt(OVERALL_SCORE, 1)} / 100")
lines.append("=" * 50)
CONSOLE_REPORT = "\n".join(lines)
print(CONSOLE_REPORT)

box = [
    "+--------------------------------------+",
    "|       DOCLING PARSING QUALITY        |",
    "+--------------------------------------+",
    f"| Layout Analysis       : {fmt(AREA_SCORES['Layout'], 1):>6} / 100 |",
    f"| Text Hierarchy        : {fmt(AREA_SCORES['Hierarchy'], 1):>6} / 100 |",
    f"| Table Structure       : {fmt(AREA_SCORES['Table'], 1):>6} / 100 |",
    "|                                      |",
    f"| Overall Score         : {fmt(OVERALL_SCORE, 1):>6} / 100 |",
    "+--------------------------------------+",
]
SCORE_BOX = "\n".join(box)
print()
print(SCORE_BOX)

        DOCLING PARSING EVALUATION
Documents scored : 8  (GT: tiger_inc/md)
Documents N/A    : 3  (no ground truth)

[1] Layout Analysis
--------------------------------------------------
Element Detection F1  : 0.93
Element Type Accuracy : 0.92
Bounding Box IoU      : N/A  (no GT coordinates)
Layout Score          : 92.6 / 100

[2] Text Hierarchy
--------------------------------------------------
Heading Detection F1  : 0.84
Heading Level (offset): 0.93   (strict: 0.15)
Parent-Child Relation : 0.74
Document Order        : 0.92
Hierarchy Score       : 85.6 / 100

[3] Table Structure
--------------------------------------------------
Table Detection F1    : 0.93
Row Accuracy          : 0.59
Column Accuracy       : 1.00
Header Accuracy       : 0.94
Cell Accuracy         : 0.96   (strict: 0.76)
Merged Cell Accuracy  : N/A  (not expressible in MD GT)
Table Score           : 89.6 / 100

OVERALL SCORE : 89.3 / 100

+--------------------------------------+
|       DOCLING PARSING QUALITY     

---
## 7. 실패 사례 및 오류 분석

지금까지 각 절에서 쌓아 둔 오류를 유형별로 집계하고 실제 사례를 본다. 유형 체계는 다음과 같다.

`Layout Error`(타입 오분류) · `Layout Missing` · `Layout Extra` · `Text Mismatch`(부분 일치) ·
`BBox Geometry` · `Heading Missing` · `Heading Extra` · `Heading Text Mismatch` ·
`Heading Level Error` · `Parent-Child Error` · `Document Order Error` ·
`Table Detection Error` · `Row/Column Error` · `Header Error` · `Cell Content Error` ·
`OCR/Text Error`(공백·자간)

In [23]:
error_df = pd.DataFrame(ERROR_CASES)
if error_df.empty:
    print("오류 0건")
else:
    error_df["Page"] = pd.to_numeric(error_df["Page"], errors="coerce").astype("Int64")
    by_type = (error_df.groupby(["Area", "Error Type"]).size()
               .reset_index(name="Count").sort_values("Count", ascending=False))
    by_doc = (error_df.groupby(["Document", "Area"]).size()
              .reset_index(name="Count").pivot(index="Document", columns="Area", values="Count")
              .fillna(0).astype(int))
    print(f"총 오류 {len(error_df)}건")
    display(by_type)
    print("문서별 오류 수")
    display(by_doc)

    print("\n" + "=" * 50)
    print("ERROR CASES (유형별 대표 사례)")
    print("=" * 50)
    for etype in by_type["Error Type"]:
        sample = error_df[error_df["Error Type"] == etype].head(2)
        for _, r in sample.iterrows():
            page = f"Page {r['Page']}" if pd.notna(r["Page"]) else "Page -"
            print(f"\n[{etype}] {r['Document']} / {page}")
            print(f"  Expected : {r['Expected']}")
            print(f"  Detected : {r['Detected']}")
            if r["Note"]:
                print(f"  Note     : {r['Note']}")

총 오류 415건


,Area,Error Type,Count
12,Table,OCR/Text Error,161
7,Layout,Layout Extra,51
6,Layout,Layout Error,32
1,Hierarchy,Heading Extra,31
5,Hierarchy,Parent-Child Error,30
10,Table,Cell Content Error,27
14,Table,Table Detection Error,16
13,Table,Row/Column Error,13
8,Layout,Layout Missing,12
3,Hierarchy,Heading Missing,11


문서별 오류 수


Area,Hierarchy,Layout,Table
Document,,,
법인카드_사용규정,15,16,8
부서소개,2,5,57
업무추진비_사용규정,12,13,1
조직도,8,9,5
조직설계_상세기획서,17,22,64
직급체계,2,4,4
출장비_사용규정,23,11,16
회식_운영규정,16,21,64



ERROR CASES (유형별 대표 사례)

[OCR/Text Error] 부서소개 / Page -
  Expected : 채용·인력운영·조직문화 관리
  Detected : 채용·인력운영· 조직문화 관리
  Note     : 표 1 (1,1) — 공백·자간

[OCR/Text Error] 부서소개 / Page -
  Expected : 채용기획/실행, 평가·보상 운영, 교육체계 설계, 조직문화 프로그램 운영
  Detected : 채용기획/실행, 평가·보상 운 영, 교육체계 설계, 조직문화 프 로그램 운영
  Note     : 표 1 (1,2) — 공백·자간

[Layout Extra] 법인카드_사용규정 / Page 1
  Expected : (없음)
  Detected : [Text] 개정이력 v1.0 제정 (조문 정합성 검토 반영) / v1.1 개정: 제2조 관리자 정의 정비, 제10조 사전승인 예외절차 신 설, 별표1 이사 겸직·대행 시 한도 적용기준 명확화
  Note     : GT에 없는 요소

[Layout Extra] 법인카드_사용규정 / Page 3
  Expected : (없음)
  Detected : [List] 인을 받아 개인 카드를 발급받을 수 있다.
  Note     : GT에 없는 요소

[Layout Error] 법인카드_사용규정 / Page 1
  Expected : Text
  Detected : Heading (Heading)
  Note     : **타이거 주식회사 (Tiger Inc.)**

[Layout Error] 법인카드_사용규정 / Page 6
  Expected : Text
  Detected : List (List)
  Note     : 이 규정의 개정은 경영지원본부의 제안으로 대표이사의 승인을 받아 시행한다.

[Heading Extra] 법인카드_사용규정 / Page 1
  Expected : (없음)
  Detected : H1 타 이 거 주 식 회 사 ( T i ge r I n c . )
  

---
## 8. 정성 평가표

자동 지표로 판정이 애매하거나 정답이 없는 항목은 **사람이 최종 판단**한다. 아래 표의
`Result`는 자동 지표에서 만든 **제안값**(`≥0.90 PASS` / `≥0.75 WARN` / 그 미만 `FAIL` /
정답 없음 `REVIEW`)일 뿐이며, 평가자가 CSV를 열어 직접 고치는 것이 최종본이다.

In [24]:
def verdict(value, hi: float = 0.90, lo: float = 0.75) -> str:
    if value is None:
        return "REVIEW"
    return "PASS" if value >= hi else ("WARN" if value >= lo else "FAIL")


qual_rows = [
    ("Layout", "영역 탐지(요소 누락/오탐)", LAYOUT_DETECT_F1,
     f"F1={fmt(LAYOUT_DETECT_F1,3)} / 누락 {int(layout_detect_df['Missing(FN)'].sum())}건"),
    ("Layout", "요소 타입 분류", LAYOUT_TYPE_ACC, f"정확도={fmt(LAYOUT_TYPE_ACC,3)}"),
    ("Layout", "Bounding Box 정확도", None, "GT 좌표 없음 — 육안 검수 필요"),
    ("Layout", "머리말/꼬리말 분리", None, f"docling이 furniture로 분리한 요소 {len(_furn)}건 — 육안 확인"),
    ("Hierarchy", "헤딩 탐지", HIER_DETECT_F1, f"F1={fmt(HIER_DETECT_F1,3)}"),
    ("Hierarchy", "헤딩 레벨(H1/H2/H3) 구분", HIER_LEVEL_ACC,
     f"offset 보정={fmt(HIER_LEVEL_ACC,3)}, strict={fmt(HIER_LEVEL_ACC_STRICT,3)}"),
    ("Hierarchy", "부모-자식 관계", HIER_PARENT_ACC, f"정확도={fmt(HIER_PARENT_ACC,3)}"),
    ("Hierarchy", "문서 순서(리딩오더)", HIER_ORDER_ACC, f"정확도={fmt(HIER_ORDER_ACC,3)}"),
    ("Table", "표 탐지", TABLE_DETECT_F1,
     f"F1={fmt(TABLE_DETECT_F1,3)} / 분할 {int(table_detect_df['Split'].sum())}건"),
    ("Table", "행/열 복원", None if TABLE_ROW_ACC is None or TABLE_COL_ACC is None
     else (TABLE_ROW_ACC + TABLE_COL_ACC) / 2,
     f"row={fmt(TABLE_ROW_ACC,3)}, col={fmt(TABLE_COL_ACC,3)}"),
    ("Table", "헤더 인식", TABLE_HEADER_ACC, f"정확도={fmt(TABLE_HEADER_ACC,3)}"),
    ("Table", "셀 값 정확도", TABLE_CELL_ACC,
     f"loose={fmt(TABLE_CELL_ACC,3)}, strict={fmt(TABLE_CELL_ACC_STRICT,3)}"),
    ("Table", "병합 셀", None, "마크다운 GT로 판정 불가 — 원본 PDF와 육안 대조 필요"),
    ("Text", "공백·자간 결함", None,
     f"공백만 다른 셀 {cell_stat.get('공백·자간', 0)}건 — CJK 양끝맞춤 조판 유래"),
]

qualitative_df = pd.DataFrame(
    [{"Category": c, "Evaluation Item": item, "Auto Metric": None if v is None else round(float(v), 3),
      "Result": verdict(v), "Comment": comment}
     for c, item, v, comment in qual_rows]
)
display(qualitative_df)
print("→ qualitative_review.csv 를 열어 Result / Comment 를 평가자가 직접 수정한다.")

,Category,Evaluation Item,Auto Metric,Result,Comment
0,Layout,영역 탐지(요소 누락/오탐),0.929,PASS,F1=0.929 / 누락 12건
1,Layout,요소 타입 분류,0.923,PASS,정확도=0.923
2,Layout,Bounding Box 정확도,NaN,REVIEW,GT 좌표 없음 — 육안 검수 필요
3,Layout,머리말/꼬리말 분리,NaN,REVIEW,docling이 furniture로 분리한 요소 147건 — 육안 확인
4,Hierarchy,헤딩 탐지,0.844,WARN,F1=0.844
5,Hierarchy,헤딩 레벨(H1/H2/H3) 구분,0.930,PASS,"offset 보정=0.930, strict=0.149"
6,Hierarchy,부모-자식 관계,0.737,FAIL,정확도=0.737
7,Hierarchy,문서 순서(리딩오더),0.921,PASS,정확도=0.921
8,Table,표 탐지,0.928,PASS,F1=0.928 / 분할 11건
9,Table,행/열 복원,0.797,WARN,"row=0.594, col=1.000"


→ qualitative_review.csv 를 열어 Result / Comment 를 평가자가 직접 수정한다.


---
## 9. 시각화

`matplotlib`이 있으면 차트를, 없으면 ASCII 막대를 그린다(`docling` env에는 기본 미설치일 수 있다).

In [25]:
def ascii_bar(label: str, value: float | None, width: int = 20) -> str:
    if value is None:
        return f"{label:<16} {'':<{width}}  N/A"
    filled = int(round(value / 100 * width))
    return f"{label:<16} {'█' * filled}{'░' * (width - filled)} {value:5.1f}"


print("[평가 점수]")
for area in ("Layout", "Hierarchy", "Table"):
    print(ascii_bar(area, AREA_SCORES[area]))
print(ascii_bar("OVERALL", OVERALL_SCORE))

print("\n[오류 유형별 건수]")
if not error_df.empty:
    counts = error_df["Error Type"].value_counts()
    top = counts.max()
    for k, v in counts.items():
        print(f"{k:<24} {'█' * max(1, int(v / top * 24))} {v}")

CHART_PATH = None
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib import font_manager

    for cand in ("Malgun Gothic", "NanumGothic", "AppleGothic"):
        if any(f.name == cand for f in font_manager.fontManager.ttflist):
            plt.rcParams["font.family"] = cand
            break
    plt.rcParams["axes.unicode_minus"] = False

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    names = ["Layout", "Hierarchy", "Table", "OVERALL"]
    vals = [AREA_SCORES["Layout"], AREA_SCORES["Hierarchy"], AREA_SCORES["Table"], OVERALL_SCORE]
    plot_names = [n for n, v in zip(names, vals) if v is not None]
    plot_vals = [v for v in vals if v is not None]
    axes[0].bar(plot_names, plot_vals, color=["#4C78A8", "#54A24B", "#E45756", "#B279A2"])
    axes[0].set_ylim(0, 100)
    axes[0].set_title("Docling Parsing Score (/100)")
    for i, v in enumerate(plot_vals):
        axes[0].text(i, v + 1.5, f"{v:.1f}", ha="center")

    if not error_df.empty:
        counts = error_df["Error Type"].value_counts().sort_values()
        axes[1].barh(counts.index, counts.values, color="#F58518")
        axes[1].set_title("Error Cases by Type")
    else:
        axes[1].axis("off")
    fig.tight_layout()
    CHART_PATH = EVAL_DIR / "evaluation_charts.png"
    fig.savefig(CHART_PATH, dpi=120)
    plt.close(fig)
    print(f"\n차트 저장: {CHART_PATH}")
except ModuleNotFoundError:
    print("\n[info] matplotlib 미설치 → ASCII 막대만 출력했다. "
          "차트가 필요하면 `conda install -n docling matplotlib`.")

[평가 점수]
Layout           ███████████████████░  92.6
Hierarchy        █████████████████░░░  85.6
Table            ██████████████████░░  89.6
OVERALL          ██████████████████░░  89.3

[오류 유형별 건수]
OCR/Text Error           ████████████████████████ 161
Layout Extra             ███████ 51
Layout Error             ████ 32
Heading Extra            ████ 31
Parent-Child Error       ████ 30
Cell Content Error       ████ 27
Table Detection Error    ██ 16
Row/Column Error         █ 13
Layout Missing           █ 12
Heading Missing          █ 11
Document Order Error     █ 9
Heading Level Error      █ 8
Text Mismatch            █ 6
Heading Text Mismatch    █ 6
Header Error             █ 2

차트 저장: D:\project\SKN29-FINAL-1TEAM\docling_eval\output\evaluation\evaluation_charts.png


---
## 10. 평가 결과 저장 및 최종 리포트

`output/evaluation/` 아래에 CSV 4종 + 마크다운 리포트를 쓴다.

In [26]:
def df_to_md(df: pd.DataFrame, max_rows: int = 60) -> str:
    """tabulate 없이 마크다운 파이프 표를 만든다."""
    if df is None or df.empty:
        return "_(데이터 없음)_"
    view = df.head(max_rows)
    cols = [str(c) for c in view.columns]
    out = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, r in view.iterrows():
        cells = []
        for c in view.columns:
            v = r[c]
            cells.append("" if pd.isna(v) else str(v).replace("|", "\\|").replace("\n", " "))
        out.append("| " + " | ".join(cells) + " |")
    if len(df) > max_rows:
        out.append(f"| … 외 {len(df) - max_rows}행 |" + " |" * (len(cols) - 1))
    return "\n".join(out)


summary_df.to_csv(EVAL_DIR / "evaluation_summary.csv", index=False, encoding="utf-8-sig")
details_df.to_csv(EVAL_DIR / "evaluation_details.csv", index=False, encoding="utf-8-sig")
error_df.to_csv(EVAL_DIR / "error_cases.csv", index=False, encoding="utf-8-sig")
qualitative_df.to_csv(EVAL_DIR / "qualitative_review.csv", index=False, encoding="utf-8-sig")

na_metrics = summary_df[summary_df["Status"] == "N/A"]["Metric"].tolist()
err_by_type = (error_df["Error Type"].value_counts().rename_axis("Error Type")
               .reset_index(name="Count") if not error_df.empty else pd.DataFrame())

improve = []
for area, metric, value, _w, _n in METRIC_SPEC:
    if value is not None and value < 0.90:
        improve.append(f"- **{area} / {metric}** = {value:.3f} — 개선 필요")
if cell_stat.get("공백·자간", 0):
    improve.append(f"- **공백·자간 결함** {cell_stat['공백·자간']}건 — CJK 양끝맞춤 조판 유래. "
                   "청크 임베딩 전 공백 재결합 후처리 필요")
if not table_detect_df.empty and table_detect_df["Split"].sum():
    improve.append(f"- **표 분할** {int(table_detect_df['Split'].sum())}건 — 페이지 경계에서 표가 쪼개짐. "
                   "인접 페이지 동일 열 구조 표 병합 후처리 필요")

report = f"""# Docling PDF 파싱 품질 평가 리포트

> 생성: `docling_eval/docling_parsing_evaluation.ipynb`

## 1. 평가 대상

- 자동 정량 평가: **{len(scored_docs)}종** — {', '.join(scored_docs)}
- N/A (Ground Truth 없음): **{len(na_docs)}종** — {', '.join(na_docs) if na_docs else '없음'}
- Ground Truth: `tiger_inc/md/*.md` (PDF와 동일 파일명의 원고)
- 평가 입력: `docling_eval/output/` (layout CSV · 표 JSON · 마크다운)

{df_to_md(pdf_info_df)}

## 2. 평가 방법

| 영역 | 방법 | 가중치 |
|---|---|---|
| Layout Analysis | GT 요소 시퀀스와 docling 요소 시퀀스를 텍스트로 정렬 → 탐지 P/R/F1 + 타입 혼동행렬 | 0.30 |
| Text Hierarchy | 헤딩 정렬 → 탐지 F1 · 레벨 정확도(오프셋 보정) · 부모-자식 · 문서 순서(LIS) | 0.30 |
| Table Structure | 셀 격자 JSON을 GT 파이프 표와 자카드 매칭 → 탐지/행/열/헤더/셀 값 | 0.40 |

정규화는 2단계(`norm_strict` = 공백 접기, `norm_loose` = 공백·구두점 제거)를 쓰며,
`loose`는 같은데 `strict`가 다른 경우를 **공백·자간 결함**으로 따로 집계한다.

**N/A로 제외한 지표**: {', '.join(na_metrics) if na_metrics else '없음'}
(정답 bbox·병합 정보가 GT에 없어 점수화하지 않았고, 가중치는 같은 영역 내에서 재분배했다.)

## 3. Layout Analysis 결과

{df_to_md(layout_detect_df)}

- 전체 Element Detection: P={fmt(LAYOUT_DETECT_P,3)} / R={fmt(LAYOUT_DETECT_R,3)} / F1={fmt(LAYOUT_DETECT_F1,3)}
- 요소 타입 분류 정확도: {fmt(LAYOUT_TYPE_ACC,3)}
- Bounding Box IoU: **N/A** (GT에 좌표 없음) — 기하 이상치 {0 if geom_df.empty else len(geom_df)}건은 별도 검수

## 4. Text Hierarchy 결과

{df_to_md(hier_df)}

- Heading Detection F1={fmt(HIER_DETECT_F1,3)} / Level(offset 보정)={fmt(HIER_LEVEL_ACC,3)}
  (strict={fmt(HIER_LEVEL_ACC_STRICT,3)}) / Parent-Child={fmt(HIER_PARENT_ACC,3)} /
  Document Order={fmt(HIER_ORDER_ACC,3)}

## 5. Table Structure 결과

{df_to_md(table_detect_df)}

{df_to_md(table_struct_df)}

- Cell Accuracy(loose)={fmt(TABLE_CELL_ACC,3)} / (strict)={fmt(TABLE_CELL_ACC_STRICT,3)}
- Merged Cell Accuracy: **N/A** (마크다운 GT는 rowspan/colspan을 표현할 수 없음)

## 6. 정량 평가 점수

{df_to_md(summary_df)}

```text
{SCORE_BOX}
```

## 7. 오류 사례

{df_to_md(err_by_type)}

{df_to_md(error_df[['Area', 'Error Type', 'Document', 'Page', 'Expected', 'Detected']] if not error_df.empty else error_df, max_rows=40)}

## 8. 정성 평가

{df_to_md(qualitative_df)}

> `Result`는 자동 지표 기반 **제안값**이다. 최종 판정은 `qualitative_review.csv`에서 평가자가 수정한다.

## 9. 종합 결과

```text
{CONSOLE_REPORT}
```

## 10. 개선이 필요한 부분

{chr(10).join(improve) if improve else '- 임계값(0.90) 미만 지표 없음'}
"""

report_path = EVAL_DIR / "evaluation_report.md"
report_path.write_text(report, encoding="utf-8")

print("저장 완료")
for p in sorted(EVAL_DIR.iterdir()):
    print(f"  {p.relative_to(BASE_DIR)}  ({p.stat().st_size:,} bytes)")

저장 완료
  output\evaluation\error_cases.csv  (63,899 bytes)
  output\evaluation\evaluation_charts.png  (52,882 bytes)
  output\evaluation\evaluation_details.csv  (11,351 bytes)
  output\evaluation\evaluation_report.md  (18,250 bytes)
  output\evaluation\evaluation_summary.csv  (1,036 bytes)
  output\evaluation\layout_manual_review.csv  (353 bytes)
  output\evaluation\qualitative_review.csv  (1,057 bytes)


---
## 최종 정리

이 노트북이 답하는 세 질문과, 답할 수 **없는** 것을 명시해 둔다.

| 질문 | 답하는 지표 |
|---|---|
| docling이 PDF의 영역을 제대로 구분하는가? | Element Detection F1 · Type Accuracy · 혼동행렬 |
| 논리적 계층 구조를 제대로 복원하는가? | Heading Detection F1 · Level Acc · Parent-Child · Document Order |
| 표의 구조와 데이터를 제대로 복원하는가? | Table Detection F1 · Row/Col/Header Acc · Cell Accuracy |

**이 노트북이 답하지 않는 것**

- **Bounding Box 정확도** — 정답 좌표가 없다. 필요하면 PDF 페이지 렌더링 위에 docling bbox를
  겹쳐 그려 사람이 보는 방식으로 가야 한다.
- **병합 셀 복원율** — 마크다운 GT의 한계. 원본 PDF 표와 육안 대조가 필요하다.
- **법령 3종의 품질** — GT가 없다. `layout_manual_review.csv`로 사람이 판단한다.

> RAG 파이프라인 관점에서는 **Table Score와 Document Order**를 먼저 본다. 표가 쪼개지거나
> 조문 순서가 뒤집히면 검색된 청크가 규정을 잘못 진술하게 되고, 그건 룰 판정 근거의 오류로
> 그대로 이어진다.